**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/fujisawa-2008"], check=True)
    os.chdir("/content/encore/entries/fujisawa-2008")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


In [ ]:
# # Reproducing Fujisawa, Amarasingham, Harrison & Buzsaki (2008) from DANDI:000067 and CRCNS pfc-2
#
# **Paper.** Fujisawa S, Amarasingham A, Harrison MT, Buzsaki G. Behavior-dependent short-term assembly dynamics in the medial prefrontal cortex. *Nat Neurosci* 11:823-833 (2008), doi:10.1038/nn.2134. The claims file (`claims/fujisawa-2008.yaml`) was written from the version of record with its supplement.
#
# **Datasets.**
# - DANDI:000067, version 0.261106.2004 (doi:10.48324/dandi.000067/0.261106.2004). The paper cites no version of either dataset, so the most recent published version is pinned; its release notes say it changes only metadata from 0.261106.0244, and the two earlier versions (0.210812.1457, 0.250624.0419) hold 28 assets against 31, so the latest is the fullest. Read: the units table (spike times, shank, cluster), the trials tables, the head LED positions and the wideband signal of the 13 working memory sessions.
# - CRCNS pfc-2 (doi:10.6080/K01V5BWK), unversioned (description version 0.7, July 2021); MD5 sums of the files read are printed in the data check. Read, streamed from the public bucket: `behaviors.zip` (the lab's summary of 12 working memory sessions), `EE0708/EE188_Behavior.mat` and `EE188_Behavior_PFC.mat` (session EE.188, not in the zip), `EE0711/EE210_Behavior_PFC.mat` (compared with the zip's copy), and `EE188_example.mat` from inside `EE0708/EE.188.tar.gz` (the paper's Figure 2a / 3 session: 117 cells and the lab's connection list).
#
# Spike times come from the NWB files, matched to the lab's cell numbers by shank and cluster (identical to the lab's spike times, data check); trials, positions and cell lists come from CRCNS, as the NWB files lack them for some sessions and give no lab cell numbers for most.
#
# **Claims.** All 33 claims of the claims file are tested except `task-performance`, blocked (data_missing): the datasets list correct trials only.
#
# | Claim | Tested |
# |---|---|
# | channel-region-link, rats-and-units, fig2-population-uniform, fig2c-active-fraction, fig2-field-size, population-vector-width, fig2d-layer-rates, fig2e-layer-discrimination, traj-diverge, side-arm-selectivity-by-layer, central-arm-selectivity, odor-nosepoke, odor-unrelated-to-stem, retrospective-coding-rare, speed-not-correlated | tested |
# | fig3-connection-rates, fig3-cell-classes, pyr-pyr-connections, nearby-pyr-int-probability, fig3b-session-connections, fig3d-hub, fig3f-distance, fig3-trajectory-by-class, no-spatial-clustering | tested |
# | fig4-position-dependence, fig5a-session-selective-connections, fig4-pair-156-136, fig5b-example-pairs, thinning-preserves-dependence, fig6-facilitation-depression, fig6b-convergent-examples, fig7-supralinear | tested |
# | task-performance | blocked, data_missing |
#
# Rats and layers: neither dataset gives a rat's layer. Every claim with values by layer is run under the claims file's readings, R-a (FF is the layer 5 rat) and R-b (each rat in turn as the layer 5 rat). Random generators are seeded per section.
#
# Units: the lab's per-session lists (627 units; data check). They are consistent with the 1 Hz "Cells" list the CRCNS description names, which the claims file treats as a variant, and they are adopted because their total is within 1% of the paper's 633 while every sorted cluster of the NWB files gives 1,373. The claims whose values depend most on the unit list also show every NWB mPFC unit as a variant.
#
# Revised after the first verification round: the unit lists are compared with spikeph and the NWB units; peak positions use both rate readings; the first/later spike test uses the shared method's jitter and assigns each significant pair to a side by the tail of its permutation distribution; head direction is checked; a one-cell jitter variant of the connection test is added.

In [ ]:
import sys, time, itertools, collections, resource
import numpy as np
import scipy.stats as st
import scipy.signal as ss
import fujisawa as fj

T0 = time.time()
LAYER_READINGS = {"R-a (FF layer 5)": "FF", "R-b1 (EE layer 5)": "EE", "R-b2 (GG layer 5)": "GG"}
def layer_of(rat, l5):
    return "L5" if rat == l5 else "L2/3"
def pct(a, b):
    return 100.0 * a / b if b else float("nan")
def frac_sel(keys, lo, hi, res=None, reading="a"):
    res = SEL if res is None else res
    if reading == "a":
        hits = [fj.seg_in(res[k]["segments"], lo, hi) for k in keys]
    else:
        g = (fj.GRID >= lo - 1e-9) & (fj.GRID <= hi + 1e-9)
        hits = [bool(res[k]["glob"][g].any()) for k in keys]
    return 100 * np.mean(hits), int(np.sum(hits)), len(keys)
def keys_layer(layer, l5, sessions=None):
    return [k for k in KEYS if layer_of(k[0][:2], l5) == layer and (sessions is None or k[0] in sessions)]

In [ ]:
crc = fj.load_crcns()
ex = fj.load_example()
S, NT, NWBU = {}, {}, {}
for s in fj.SESSIONS:
    f = fj.open_nwb(s)
    U = fj.read_nwb_units(f)
    NT[s] = fj.read_nwb_trials(f)
    leds = fj.read_nwb_leds(f)
    f.close()
    S[s] = fj.Session(s, crc[s], U, leds, example=ex if s == "EE188" else None)
    # every mPFC unit of the NWB file (all sorted clusters), kept for the unit-list variant
    pf = np.nonzero(U["region"] == "mPFC")[0]
    NWBU[s] = dict(times=[np.asarray(U["times"][i], float) for i in pf], shank=U["shank"][pf],
                   listed=np.isin(pf, S[s].nwb_index), n_ca1=int(np.sum(U["region"] == "CA1")),
                   cell_type=U["cell_type"][pf], lab=np.asarray(U["lab"][pf]).astype(int))
print(f"loaded {len(S)} sessions in {time.time() - T0:.0f} s")

In [ ]:
print("DANDI:000067 version", fj.DANDI_VERSION, "; CRCNS pfc-2 (unversioned), MD5 of files read:")
for k, v in fj.CRCNS_MD5.items():
    print("  ", k, v)
print("\nSession  rat  units  pyr/int/unk(lab)  NWB-CRCNS identical  trials  L  R  NWB trials equal  duration(min)")
for s, x in S.items():
    nt = NT[s]
    eq = "no NWB trials" if nt is None else bool(len(nt["start"]) == x.n_trials and np.allclose(nt["start"], x.start)
                                                  and np.allclose(nt["stop"], x.stop) and np.allclose(nt["poke"], x.poke_start)
                                                  and np.all((nt["side"] == "left") == (x.side == "L")))
    c = np.bincount(x.lab_type, minlength=4)[1:]
    print(f"{s:6s} {x.rat:3s} {len(x.cells):5d}   {c[0]:3d}/{c[1]:3d}/{c[2]:3d}        {x.n_identical:3d}               {x.n_trials:3d}  "
          f"{(x.side=='L').sum():2d} {(x.side=='R').sum():2d}  {str(eq):14s} {x.duration/60:6.1f}")
print("total units", sum(len(x.cells) for x in S.values()), "; by rat",
      {r: sum(len(x.cells) for x in S.values() if x.rat == r) for r in "EFG" for r in [r * 2]})
# the EE210 summary is in behaviors.zip and in the session folder: are they the same?
a, b = crc["EE210"], crc["_EE210_folder"]
print("EE210 behaviors.zip copy identical to folder copy:",
      all(np.array_equal(a[k], b[k]) for k in b if not k.startswith("__")))
# the Figure 2a session: example file against the session's Behavior_PFC
pf = crc["EE188"]["CellsInfo"]
exc = ex["CellsInfo"]
print("EE188: example file cells", len(exc), "; Behavior_PFC cells", len(pf), "; common", len(set(pf[:, 0]) & set(exc[:, 0])))
same = {c: (t1, t2) for c, t1 in zip(pf[:, 0], pf[:, 2]) for c2, t2 in zip(exc[:, 0], exc[:, 2]) if c == c2}
agree = sum(t1 == t2 for t1, t2 in same.values())
print("  lab cell type agrees between the two files for", agree, "of", len(same), "common cells;",
      "crosstab (Behavior_PFC code, example code):", collections.Counter(same.values()))
print("  shank in example CellsInfo equals spikeph shank:",
      np.array_equal(exc[:, 1].astype(int), S["EE188"].shank))
# ranges of variables used
for s, x in S.items():
    d = x.d[x.d > 0]
    tr = x.trial_of(x.whl_t)
    inside = (x.d > 0)
    print(f"{s}: d in [{d.min():.3f}, {d.max():.3f}]; whl frames with d>0 outside SessionNP trials "
          f"{np.mean(tr[inside] < 0):.3f}; trial frames with d=0 {np.mean(x.d[tr >= 0] == 0):.3f}; "
          f"trial duration {np.median(x.stop - x.start):.1f} s (median); poke {np.median(x.start - x.poke_start):.1f} s; "
          f"LED px x [{np.nanmin(x.leds[0][:,0]):.0f},{np.nanmax(x.leds[0][:,0]):.0f}] y [{np.nanmin(x.leds[0][:,1]):.0f},{np.nanmax(x.leds[0][:,1]):.0f}]; "
          f"LED frames {len(x.leds[0])} vs whl {len(x.whl)}")
# do the NWB LEDs equal the whl columns?
for s, x in S.items():
    w = x.whl[:, :2].copy(); w[w < 0] = np.nan
    n = min(len(w), len(x.leds[0]))
    print(s, "NWB led_1 equals whl x1,y1 where both tracked:",
          np.allclose(np.nan_to_num(w[:n], nan=-9), np.nan_to_num(x.leds[0][:n], nan=-9), atol=1e-6), end="; ")
print()

In [ ]:
# Spike positions: the lab's spikeX against positions interpolated here from the whl file
m = crc["EE049"]
x = S["EE049"]
c0 = x.cells[0]
sel = m["spikeind"].ravel() == c0
sx = m["spikeX"].ravel()[sel]
tt = m["spiket"].ravel()[sel] / fj.FS
ok = sx >= 0
xi = np.interp(tt[ok], x.whl_t, x.d) - 1
print("EE049 cell", c0, ": spikes with spikeX >= 0:", ok.sum(), "; in SessionNP trials here:",
      (x.trial_of(tt) >= 0).sum(), "; max |spikeX - interpolated d-1|", np.abs(xi - sx[ok]).max())
# Clock check: decode the trial side from population spike counts in the 2 s after trial start,
# against the same with trial times shifted by 60 s (labels kept), leave-one-out nearest centroid.
def decode(x, shift):
    X = np.array([[np.sum((t >= a + shift) & (t < a + shift + 2.0)) for t in x.times] for a in x.start], float)
    X = (X - X.mean(0)) / (X.std(0) + 1e-9)
    y = x.side == "L"
    hit = 0
    for i in range(len(y)):
        m = np.ones(len(y), bool); m[i] = False
        cL, cR = X[m & y].mean(0), X[m & ~y].mean(0)
        hit += (np.sum((X[i] - cL) ** 2) < np.sum((X[i] - cR) ** 2)) == y[i]
    return hit / len(y)
print("side decoding accuracy, 0-2 s after trial start (true times / times shifted +60 s), binomial p of the true-time accuracy against chance:")
WEAK_CLOCK = []
for s, x in S.items():
    a0, a60 = decode(x, 0), decode(x, 60)
    ch = max(np.mean(x.side == 'L'), np.mean(x.side == 'R'))
    p = st.binomtest(int(round(a0 * x.n_trials)), x.n_trials, ch, alternative="greater").pvalue
    if p >= 0.05 or a0 <= a60:
        WEAK_CLOCK.append(s)
    print(f"  {s}: {a0:.2f} / {a60:.2f}  (n trials {x.n_trials}, chance {ch:.2f}; p = {p:.3g})")
print("sessions where this check does not establish the shared clock (p >= 0.05, or not above the shifted times):", WEAK_CLOCK)

In [ ]:
# Unit lists (claims file header, "Units"). The lab's per-session list (CellsInfo of the Behavior_PFC files; for EE.188
# the example file's 117 cells) is used for every claim. The CRCNS description documents a variable "Cells" (units whose
# maximum rate in the trials is above 1 Hz) but not CellsInfo of the Behavior_PFC files. Compare the lists with spikeph
# (every unit the lab sorted) and with every mPFC unit of the NWB files.
print("Behavior_PFC variables (EE049):", sorted(k for k in crc["EE049"] if not k.startswith("__")),
      "; a variable 'Cells' in any session:", any("Cells" in crc[s] for s in fj.SESSIONS))
print("\nSession  spikeph  NWB mPFC/CA1  lab list  in spikeph  units with spikes in spiket | max trial rate >= 1 Hz: listed / not listed | median max rate (Hz): listed / not listed")
MAXR = {}
for s, x in S.items():
    m = crc[s]; sp = m.get("spikeph_all", m["spikeph"])
    O = fj.occupancy(x)
    r = np.array([fj.max_trial_rate(fj.trial_psth_times(x, t), O, x.side) for t in NWBU[s]["times"]])
    li = NWBU[s]["listed"]
    MAXR[s] = (r, li)
    nsp = len(np.unique(m["spikeind"]))
    print(f"{s}   {len(sp):4d}     {len(li):4d} / {NWBU[s]['n_ca1']:3d}    {li.sum():4d}      {int(np.isin(x.cells, sp[:, 0]).sum()):4d}       {nsp:4d}"
          f"                     | {np.mean(r[li] >= 1):.2f} / {np.mean(r[~li] >= 1) if (~li).any() else np.nan:.2f}"
          f" ({int(np.sum(r[~li] >= 1))} of {int((~li).sum())}) | {np.median(r[li]):.2f} / {np.median(r[~li]) if (~li).any() else np.nan:.2f}")
r_all = np.concatenate([v[0] for v in MAXR.values()]); l_all = np.concatenate([v[1] for v in MAXR.values()])
print(f"all sessions: listed units with max trial rate >= 1 Hz {np.mean(r_all[l_all] >= 1):.3f} ({int(np.sum(r_all[l_all] < 1))} below);"
      f" units not listed {np.mean(r_all[~l_all] >= 1):.3f} ({int(np.sum(r_all[~l_all] >= 1))} of {int((~l_all).sum())} at or above)")
x = S["EE188"]; pf = set(crc["EE188"]["CellsInfo"][:, 0])
ext = [i for i, c_ in enumerate(x.cells) if c_ not in pf]
rr = np.array([fj.max_trial_rate(fj.trial_psth(x, i), fj.occupancy(x), x.side) for i in range(len(x.cells))])
print(f"EE.188: the example file's 117 cells hold the 105 of EE188_Behavior_PFC.mat and {len(ext)} more; max trial rate of those {len(ext)}: "
      f"{np.round(np.sort(rr[ext]), 2).tolist()} Hz; of the 105: below 1 Hz {int(np.sum(rr[[i for i in range(len(x.cells)) if i not in ext]] < 1))}")
tot_l = {r_: sum(len(S[s].cells) for s in S if s[:2] == r_) for r_ in ("EE", "FF", "GG")}
tot_n = {r_: sum(len(NWBU[s]["listed"]) for s in S if s[:2] == r_) for r_ in ("EE", "FF", "GG")}
print("units per rat, lab lists:", tot_l, sum(tot_l.values()), "; every NWB mPFC unit:", tot_n, sum(tot_n.values()), "(paper 633)")
# Spike times identical to the lab's (spiket of the Behavior_PFC files), per session; EE.188's 12 extra cells have no lab spikes
print("cells with spike times identical to the lab's:", sum(x.n_identical for x in S.values()), "of", sum(len(x.cells) for x in S.values()),
      "; EE.188", S["EE188"].n_identical, "of", len(S["EE188"].cells), "(the 12 cells not in EE188_Behavior_PFC.mat have no lab spikes to compare)")
# The NWB units' cell_type and lab_cell_number columns (mPFC units)
for s in S:
    u = NWBU[s]
    print(f"{s}: NWB cell_type {dict(collections.Counter(u['cell_type'].tolist()))}; lab_cell_number -1 in {int(np.sum(u['lab'] == -1))} of {len(u['lab'])}", end="")
    if np.any(u["lab"] >= 0):
        typed = np.isin(u["cell_type"], ["pyramidal cell", "interneuron"])
        ci = crc[s]["CellsInfo"]; code = dict(zip(ci[:, 0].astype(int), ci[:, 2].astype(int)))
        ty = dict(zip(u["lab"].tolist(), u["cell_type"].tolist()))
        print(f"; units with a type {int(typed.sum())}, their lab numbers equal to the Behavior_PFC CellsInfo list ({len(code)} cells): "
              f"{set(u['lab'][typed].tolist()) == set(code)}; (CellsInfo code, NWB type): {dict(collections.Counter((code[c_], ty[c_]) for c_ in code))}", end="")
        if s == "EE188":
            print(f"; NWB type of the example file's 12 other cells: {dict(collections.Counter(ty[c_] for c_ in S[s].cells.tolist() if c_ not in code))}", end="")
    print()
# The example file's type codes (1 pyr, 2 int, 3 unknown) for the cells the paper's figures name, and its pairs_all rows from 32 and 72
codes = dict(zip(ex["CellsInfo"][:, 0].astype(int), ex["CellsInfo"][:, 2].astype(int)))
print("EE188_example.mat CellsInfo codes:", {c_: codes[c_] for c_ in (32, 72, 52, 136)})
pa_ = ex["pairs_all"].astype(int)
print("pairs_all rows (pre, post) with 32 or 72 as presynaptic:", [tuple(r[:2]) for r in pa_ if r[0] in (32, 72)])

In [ ]:
# Positions within trials, and the frames with d > 0 outside the listed trials (CRCNS description: d = 1 at the end of
# the last nose poke, 2 just before reward consumption, 0 when the rat is not in a working memory trial)
print("Session: first and last tracked frame of each trial (median lag from trial start / to trial end; position there) | "
      "whl trial numbers | frames with d > 0 outside trials: share inside the trial's nose poke (first poke start to trial start), position")
for s, x in S.items():
    lag0, lag1, dfirst, dlast = [], [], [], []
    for a, b in zip(x.start, x.stop):
        fr = np.nonzero((x.whl_t >= a) & (x.whl_t <= b) & (x.d > 0))[0]
        lag0.append(x.whl_t[fr[0]] - a); lag1.append(b - x.whl_t[fr[-1]])
        dfirst.append(x.d[fr[0]] - 1); dlast.append(x.d[fr[-1]] - 1)
    tr = x.trial_of(x.whl_t)
    out = (x.d > 0) & (tr < 0)
    inpoke = np.zeros(len(x.d), bool)
    for a, b in zip(x.poke_start, x.start):
        inpoke |= (x.whl_t >= a) & (x.whl_t < b)
    nums = x.whl[:, 5]; tn = np.unique(nums[nums > 0])
    print(f"  {s}: start lag {np.median(lag0):.2f} s (max {np.max(lag0):.2f}), position {np.median(dfirst):.3f} [{np.min(dfirst):.3f}, {np.max(dfirst):.3f}]; "
          f"end lag {np.median(lag1):.2f} s (max {np.max(lag1):.2f}), position {np.median(dlast):.3f} [{np.min(dlast):.3f}, {np.max(dlast):.3f}] | "
          f"numbers {int(tn.min())} to {int(tn.max())}, {len(tn)} distinct, {int(tn.max() - tn.min() + 1 - len(tn))} missing | "
          f"{out.sum()} frames, {np.mean(inpoke[out]):.2f} in the nose poke, with a whl trial number {np.mean(nums[out] > 0):.2f}, position median {np.median(x.d[out]) - 1:.3f}")

## Claim channel-region-link

> The recording channels of DANDI:000067 can be assigned to the prefrontal probe, the CA1 probe and the nose-poke sensors by the channel map of the CRCNS pfc-2 description of the same recordings (channels 1 to 64 prefrontal, 65 to 96 CA1, 97 right and 98 left nose-poke sensors), and the data agree with that map.

Criterion: in each working memory session, (i) the median theta ratio (6 to 10 Hz over 1 to 40 Hz power, during trials) of the CA1 channels is above that of channels 1 to 64; (ii) in at least 90% of trials the sensor channel with more activity during the nose poke is the one of the trial's side. The raw wideband signal of the NWB files is read: for (i) the 52 s chunk holding the most trial time, all 96 neural channels; for (ii) channels 97 and 98 over the whole session, reduced as read to a 1 kHz envelope (the mean absolute deviation from the channel's median in 1 ms blocks). Activity during the nose poke is the mean envelope from the first poke start to the end of the last poke (SessionNP columns 1 and 2).

In [ ]:
def raw_checks(sess):
    f = fj.open_nwb(sess.name)
    labels = [x.decode() for x in f["general/extracellular_ephys/electrodes/signal"][96:]]
    if "acquisition/ElectricalSeries" not in f:
        f.close()
        return None, labels
    d = f["acquisition/ElectricalSeries/data"]
    ch = d.chunks[0]
    nchunk = int(np.ceil(d.shape[0] / ch))
    # (i) chunk with the most trial time
    tt = np.arange(nchunk) * ch / fj.FS
    cover = [np.sum(np.clip(np.minimum(sess.stop, t + ch / fj.FS) - np.maximum(sess.start, t), 0, None)) for t in tt]
    c = int(np.argmax(cover))
    X = d[c * ch:(c + 1) * ch, 0:96].astype(np.float32)
    X = ss.decimate(X, 16, axis=0, zero_phase=True)      # 1250 Hz
    fs = fj.FS / 16
    t = c * ch / fj.FS + np.arange(len(X)) / fs
    k = sess.trial_of(t)
    ratios = []
    segs = [np.nonzero(k == j)[0] for j in np.unique(k[k >= 0])]
    segs = [g for g in segs if len(g) >= 2 * fs]
    P = 0
    for g in segs:
        fr, p = ss.welch(X[g], fs=fs, nperseg=int(2 * fs), axis=0)
        P = P + p
    th = P[(fr >= 6) & (fr <= 10)].sum(0) / P[(fr >= 1) & (fr <= 40)].sum(0)
    # (ii) sensor envelope over the whole session
    raw = []
    for i in range(nchunk):
        Y = d[i * ch:(i + 1) * ch, 96:]                      # every channel from 97 on (one chunk column), int16
        n = len(Y) // 20 * 20
        raw.append(Y[:n])
    f.close()
    raw = np.concatenate(raw)
    med = np.median(raw[::1000].astype(np.float32), axis=0)
    E = np.concatenate([np.abs(raw[i:i + 20 * 2**16].astype(np.float32) - med).reshape(-1, 20, raw.shape[1]).mean(axis=1)
                        for i in range(0, len(raw), 20 * 2**16)])   # 1 kHz envelope, channels 97 on
    del raw
    return (th, len(segs), E), labels

LINK = {}
for s, x in S.items():
    r, labels = raw_checks(x)
    print(s, "NWB channel labels 97 and up:", labels)
    LINK[s] = r
rows = []
for s, x in S.items():
    if LINK[s] is None:
        print(f"{s}: no raw signal in the NWB file; not checked")
        continue
    th, nseg, E = LINK[s]
    pfc, ca1 = np.median(th[:64]), np.median(th[64:96])
    actall = np.array([E[int(a * 1000):int(b * 1000)].mean(0) for a, b in zip(x.poke_start, x.start)])
    act = actall[:, :2]
    pick = np.where(act[:, 0] > act[:, 1], "R", "L")
    # search every channel from 97 on for the one most active on right and on left pokes
    z = (actall - actall.mean(0)) / (actall.std(0) + 1e-9)
    cR = int(np.argmax(z[x.side == "R"].mean(0) - z[x.side == "L"].mean(0)))
    cL = int(np.argmax(z[x.side == "L"].mean(0) - z[x.side == "R"].mean(0)))
    pick2 = np.where(actall[:, cR] / np.median(actall[:, cR]) > actall[:, cL] / np.median(actall[:, cL]), "R", "L")
    print(f"   {s}: channel search: right sensor ch {97 + cR}, left sensor ch {97 + cL}; agreement with side {np.mean(pick2 == x.side):.2f}")
    LINK[s] = (th, nseg, E[:, :2].copy())   # keep channels 97 and 98 only
    agree = np.mean(pick == x.side)
    agree_swapped = np.mean(pick != x.side)
    rows.append((s, pfc, ca1, ca1 > pfc, agree, agree_swapped, nseg))
    print(f"{s}: theta ratio median PFC ch1-64 {pfc:.3f}, CA1 ch65-96 {ca1:.3f} (CA1 higher: {ca1 > pfc}; {nseg} trial segments >= 2 s); "
          f"sensor agreement with side {agree:.2f} (swapped map {agree_swapped:.2f}); "
          f"poke activity R-sensor/L-sensor median on R trials {np.median(act[x.side=='R',0]):.1f}/{np.median(act[x.side=='R',1]):.1f}, "
          f"on L trials {np.median(act[x.side=='L',0]):.1f}/{np.median(act[x.side=='L',1]):.1f}")
ok_i = all(r[3] for r in rows)
ok_ii = all(r[4] >= 0.9 for r in rows)
print("(i) holds in every session:", ok_i, "; (ii) holds in every session:", ok_ii)
print("peak memory so far %.1f GB" % (resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6))
print("sessions failing (ii):", [(r[0], round(r[4], 2)) for r in rows if r[4] < 0.9])

**Result: not reproduced.** (i) holds in all 12 sessions with a wideband signal: the CA1 channels' median theta ratio is above that of channels 1 to 64 in each. (ii) holds in every EE and FF session with a signal (sensor agreement with the trial's side 1.00), but fails in GG.069, GG.097 and GG.149 (0.30 to 0.42), and no channel from 97 on carries a sensor-like signal there (the channel search agrees with the side in at most 0.78 of trials, and the poke activity of channels 97 and 98 is the same on left and right trials). So no per-rat correction exists for GG; the NWB files of GG label channels 97 to 101 'neural'. EE.088 has no wideband signal in DANDI:000067 and could not be checked. The NWB channel labels from 97 on differ by rat: EE and FF files carry the same eight labels (sensors on 97 and 98, encoders on 101 and 102, sync on 104), so the FF files follow rat EE's 104-channel layout and not the 101 channels the CRCNS description gives FF; the GG files have five channels from 97 on, all labelled 'neural'.

## Claim rats-and-units

> A total of 633 well-isolated units were recorded from anterior cingulate and dorsal prelimbic cortex in four rats, 411 in layer 2/3 (three rats) and 222 in layer 5 (one rat).

Units are the cells the lab lists for each working memory session (CellsInfo of the session's Behavior_PFC file; for EE.188 the 117 cells of EE188_example.mat, the paper's Figure 2a session), each session counted apart. Every mPFC unit of the NWB files (every sorted cluster) is shown as a variant. The data check above shows that the lab's lists are consistent with the 1 Hz "Cells" list of the CRCNS description (nearly every listed unit reaches 1 Hz in the trials, most units left out do not), which the claims file names only as a variant; they are used as the paper's units because their total (627) is within 1% of the paper's 633, while every sorted cluster gives 1,373. This is an assumption, and the claims whose values depend most on the unit list (fractions of neurons, active fraction, mean rates) also show the all-unit variant.

In [ ]:
by_rat = collections.Counter()
nwb_by_rat = collections.Counter()
for s, x in S.items():
    by_rat[x.rat] += len(x.cells)
    nwb_by_rat[x.rat] += len(NWBU[s]["times"])
tot = sum(by_rat.values())
print("units per rat (lab cell lists):", dict(by_rat), "; total", tot, "(paper 633 in four rats)")
for name, l5 in LAYER_READINGS.items():
    l23 = sum(v for r, v in by_rat.items() if r != l5)
    print(f"  {name}: layer 5 rat {l5} {by_rat[l5]} units (expected 222 +/- 22.2: {abs(by_rat[l5]-222) <= 22.2}); "
          f"the other two rats {l23} (limit 452.1: {l23 <= 452.1})")
print("total within 633 +/- 63.3:", abs(tot - 633) <= 63.3)
print("EE.188 Behavior_PFC list instead of the example file: total", tot - 117 + 105)
ntot = sum(nwb_by_rat.values())
print("variant, every mPFC unit of the NWB files (all sorted clusters):", dict(nwb_by_rat), "; total", ntot,
      "; within 633 +/- 63.3:", abs(ntot - 633) <= 63.3, "; a rat with 222 +/- 22.2:", any(abs(v - 222) <= 22.2 for v in nwb_by_rat.values()))
print("largest session, lab lists:", max(len(x.cells) for x in S.values()), "; NWB mPFC units per session:",
      {s: len(NWBU[s]["times"]) for s in S}, "; a session of 142 units in either list:",
      any(len(x.cells) == 142 or len(NWBU[s]["times"]) == 142 for s, x in S.items()))

**Result: partially reproduced (resting on an assumption: the unit list).** The lab's cell lists hold 627 units, within 633 +/- 63.3, in three rats (EE 446, FF 146, GG 35). No rat holds 222 +/- 22.2 units under any rat reading, so the layer 5 count does not match. With every mPFC unit of the NWB files the total is 1,373, outside 633 +/- 63.3, and no rat holds 222 +/- 22.2 either (FF 162), so under that list the outcome would be not reproduced. The partial outcome therefore rests on taking the lab's lists as the paper's units, and that check is partly circular: the lists were adopted in part because their total is near 633. No session has 142 units in either list. The lab's lists are consistent with a 1 Hz rate rule (98.9% of listed units reach 1 Hz in the trials, against 28.7% of the units left out; data check).

## Claim task-performance

> Rats performed the odor-place matching task at 92% correct, 91.9% in the recording sessions.

The criterion asks for correct over all trials and blocks the claim when the datasets list correct trials only. SessionNP is described as "Only correct trials are listed", and the NWB trials tables hold the same trials (data check above). The linearised position file numbers only the same trials.

In [ ]:
for s, x in S.items():
    w = x.whl
    nums = np.unique(w[w[:, 6] > 0, 5])
    nums = nums[nums > 0]
    print(f"{s}: SessionNP trials {x.n_trials}; trial numbers in the whl file {len(nums)}; NWB trials "
          f"{'none' if NT[s] is None else len(NT[s]['start'])}")
print("No error trials are listed in either dataset: the claim is blocked (data_missing).")

**Result: blocked (data_missing).** Every session lists the same trials in SessionNP, the whl file's trial numbers and the NWB trials table, and the CRCNS description says only correct trials are listed; error trials are not in either dataset.

## Left-right permutation tests of every neuron (shared by several claims)

The Methods' test (claims file, shared method): per neuron, the PSTH of each condition is the sum of gaussian kernels (bandwidth 0.05 normalised length) over its spikes divided by its number of trials; D0 = left minus right; labels permuted M = 1,000 times; two-sided pointwise bands and global bands by the stepping procedure at alpha = 0.05; a segment is significant when the global band is broken, and its extent is the run of pointwise breaks. Positions: the lab's linearised position d - 1 (0 at the end of the last nose poke, 1 at the start of reward consumption; CRCNS description), correct trials (the only ones listed). So of the claims file's readings, N-a, E-b and T-a are the only ones the data allow. The same test is run with labels from the previous listed trial (Supplementary Figure 7), and once with one random relabelling of each neuron's trials as a null. The next cell prints the band index the stepping procedure settles on and its pointwise-equivalent level, and runs the same test on every NWB mPFC unit the lab's lists leave out (unit-list variant).

In [ ]:
SEL, PREV, NULL1 = {}, {}, {}
PSTH = {}
for si, (s, x) in enumerate(S.items()):
    rng = np.random.default_rng(1000 + si)
    lab = x.side == "L"
    prev = np.r_[False, lab[:-1]]
    for u in range(len(x.cells)):
        P = fj.trial_psth(x, u)
        PSTH[(s, u)] = P
        SEL[(s, u)] = fj.perm_bands(P, lab, M=1000, alpha=0.05, rng=rng)
        PREV[(s, u)] = fj.perm_bands(P[1:], prev[1:], M=1000, alpha=0.05, rng=rng)
        NULL1[(s, u)] = fj.perm_bands(P, rng.permutation(lab), M=1000, alpha=0.05, rng=rng)
OCC = {s: fj.occupancy(x) for s, x in S.items()}
KEYS = list(SEL)
print("neurons tested", len(KEYS), "; neurons without spikes in trials",
      sum(PSTH[k].sum() == 0 for k in KEYS), f"; {time.time() - T0:.0f} s")

In [ ]:
# Global band level of the left-right tests: kstar is the band index (rank from either end) that the stepping procedure
# settles on; its pointwise-equivalent two-sided level is 2 kstar / (M + 1) (the paper's example: global 5% about pointwise 0.6%)
ks = np.array([SEL[k]["kstar"] for k in KEYS])
lv = 2 * ks / 1001
print(f"left-right tests, M = 1,000, alpha = 0.05: kstar median {np.median(ks):.0f} (range {ks.min()} to {ks.max()}); "
      f"pointwise-equivalent level median {100 * np.median(lv):.2f}% (5th to 95th percentile {100 * np.percentile(lv, 5):.2f}% to {100 * np.percentile(lv, 95):.2f}%)")
# Variant on the unit list (data check): the same test for every NWB mPFC unit the lab's lists leave out
SEL_X = {}
t1 = time.time()
for si, (s, x) in enumerate(S.items()):
    rng = np.random.default_rng(3000 + si)
    lab = x.side == "L"
    for i in np.nonzero(~NWBU[s]["listed"])[0]:
        SEL_X[(s, -1 - int(i))] = fj.perm_bands(fj.trial_psth_times(x, NWBU[s]["times"][i]), lab, M=1000, alpha=0.05, rng=rng)
SEL_ALL = {**SEL, **SEL_X}
KEYS_ALL = KEYS + list(SEL_X)
print(f"unit-list variant: {len(SEL_X)} NWB mPFC units not in the lab's lists tested in {time.time() - t1:.0f} s; "
      f"significant anywhere {pct(sum(v['significant'] for v in SEL_X.values()), len(SEL_X)):.1f}% (lab-listed units "
      f"{pct(sum(SEL[k]['significant'] for k in KEYS), len(KEYS)):.1f}%)")
def keys_layer_all(layer, l5):
    return [k for k in KEYS_ALL if layer_of(k[0][:2], l5) == layer]

## Claim side-arm-selectivity-by-layer

> In the side arms (positions 0.5 to 1) the fraction of neurons firing differently on left and right trials is almost twice as large in layer 5 (40%) as in layer 2/3 (22%).

Pooled over all sessions of each layer, under each rat reading. Reading (a): any part of a significant segment lies in 0.5 to 1; (b): the global band is broken within 0.5 to 1.

In [ ]:
SIDE = {}
for name, l5 in LAYER_READINGS.items():
    for rd in "ab":
        f5, n5, N5 = frac_sel(keys_layer("L5", l5), 0.5, 1, reading=rd)
        f2, n2, N2 = frac_sel(keys_layer("L2/3", l5), 0.5, 1, reading=rd)
        p = st.fisher_exact([[n5, N5 - n5], [n2, N2 - n2]], alternative="greater")[1]
        SIDE[(name, rd)] = (f5, f2, p)
        print(f"{name}, reading ({rd}): layer 5 {f5:.1f}% ({n5}/{N5}; expected 40 +/- 4: {abs(f5-40)<=4}), "
              f"layer 2/3 {f2:.1f}% ({n2}/{N2}; expected 22 +/- 2.2: {abs(f2-22)<=2.2}); one-sided Fisher L5 > L2/3 p = {p:.3g}")
print("per session, reading (a):", {s: round(frac_sel([k for k in KEYS if k[0] == s], 0.5, 1)[0], 1) for s in S})
print("null (one random relabelling per neuron), all neurons, reading (a): %.1f%%" % frac_sel(KEYS, 0.5, 1, res=NULL1)[0])
print("variant, every NWB mPFC unit (unit list, data check):")
for name, l5 in LAYER_READINGS.items():
    for rd in "ab":
        f5, n5, N5 = frac_sel(keys_layer_all("L5", l5), 0.5, 1, res=SEL_ALL, reading=rd)
        f2, n2, N2 = frac_sel(keys_layer_all("L2/3", l5), 0.5, 1, res=SEL_ALL, reading=rd)
        p = st.fisher_exact([[n5, N5 - n5], [n2, N2 - n2]], alternative="greater")[1]
        print(f"   {name} ({rd}): layer 5 {f5:.1f}% ({n5}/{N5}; 40 +/- 4: {abs(f5-40)<=4}), layer 2/3 {f2:.1f}% ({n2}/{N2}; 22 +/- 2.2: {abs(f2-22)<=2.2}); Fisher p = {p:.3g}")

**Result: partially reproduced.** Under R-a (FF layer 5), layer 5 is above layer 2/3 (61.0% against 52.6%, one-sided Fisher p = 0.046, reading (a); 60.3% against 49.9%, p = 0.017, reading (b)), but both fractions are far above the paper's 40% and 22%. Under R-b neither rat assignment gives layer 5 above layer 2/3. With one random relabelling of each neuron's trials, 3.3% of neurons are selective in the side arms, so the test's false positive rate is near its 5% level. With every NWB mPFC unit the layer 2/3 fraction falls to 33.8% (409/1,211, reading (a)) and layer 5 is 58.6% (Fisher p = 1.2e-09), still outside both tolerances, so the outcome is the same under both unit lists. These variant counts move by one unit between runs of the same seeded code: the round-2 verifier's executed copy printed 33.7% (408/1,211) and p = 1.1e-09.

## Claim central-arm-selectivity

> A smaller fraction of neurons, 16% in layer 2/3 and 18% in layer 5, fire differently on left and right trials already in the central arm (positions 0 to 0.3), where paths and head directions are apparently indistinguishable.

In [ ]:
for name, l5 in LAYER_READINGS.items():
    for rd in "ab":
        f5, n5, N5 = frac_sel(keys_layer("L5", l5), 0, 0.3, reading=rd)
        f2, n2, N2 = frac_sel(keys_layer("L2/3", l5), 0, 0.3, reading=rd)
        print(f"{name}, reading ({rd}): layer 2/3 {f2:.1f}% ({n2}/{N2}; expected 16 +/- 1.6: {abs(f2-16)<=1.6}), "
              f"layer 5 {f5:.1f}% ({n5}/{N5}; expected 18 +/- 1.8: {abs(f5-18)<=1.8})")
fa, na, Na = frac_sel(KEYS, 0, 0.3)
fn, nn, Nn = frac_sel(KEYS, 0, 0.3, res=NULL1)
p = st.fisher_exact([[na, Na - na], [nn, Nn - nn]], alternative="greater")[1]
print(f"pooled: {fa:.1f}% ({na}/{Na}) against one random relabelling {fn:.1f}% ({nn}/{Nn}); one-sided Fisher p = {p:.3g}")
print("per session:", {s: round(frac_sel([k for k in KEYS if k[0] == s], 0, 0.3)[0], 1) for s in S})
print("variant, every NWB mPFC unit (unit list, data check):")
for name, l5 in LAYER_READINGS.items():
    for rd in "ab":
        f5, n5, N5 = frac_sel(keys_layer_all("L5", l5), 0, 0.3, res=SEL_ALL, reading=rd)
        f2, n2, N2 = frac_sel(keys_layer_all("L2/3", l5), 0, 0.3, res=SEL_ALL, reading=rd)
        print(f"   {name} ({rd}): layer 2/3 {f2:.1f}% ({n2}/{N2}; 16 +/- 1.6: {abs(f2-16)<=1.6}), layer 5 {f5:.1f}% ({n5}/{N5}; 18 +/- 1.8: {abs(f5-18)<=1.8})")

**Result: partially reproduced.** 30.1% of neurons (189/627) have a significant segment in 0 to 0.3, against 1.4% after one random relabelling (one-sided Fisher p = 6e-52); by layer the fractions are 22.9% to 30.7% under every rat reading and reading, outside 16 +/- 1.6 and 18 +/- 1.8. With every NWB mPFC unit some single values fall within tolerance (R-a, layer 2/3 17.2% and 15.2%, the latter 15.3% in the round-2 verifier's executed copy; R-b2 (b), 16.2% and 17.9%, both within), but that unit list holds 1,373 units, more than twice the paper's 633, so it is shown as a variant and not taken as the paper's reading.

## Monosynaptic connections by the jitter method (shared by the connection claims)

Methods, "Identifying monosynaptic interactions": each spike of both neurons jittered uniformly on [-5, +5] ms, 1,000 surrogates; cross-correlogram over [-5, +5] ms; the global band from the maximum and minimum of each surrogate at the 99% level; a connection is excitatory (peak) or inhibitory (trough) when the counts break the global band anywhere in [1, 4] ms; pairs counted in both directions; the whole task session (intertrial intervals included). Bin readings B-a 1 ms and B-b 0.5 ms (the 1 ms CCG is the sum of two 0.5 ms bins of the same surrogates); width readings W-a (the global band test alone) and W-b (also the run of bins above the pointwise 99% band around the peak spans 2 ms or less). As a variant (claims file header; Supplementary Figure 9A), the same test with the spikes of one neuron of each pair jittered (B-a, W-a).

In [ ]:
COMBOS = [("B-a", "W-a"), ("B-a", "W-b"), ("B-b", "W-a"), ("B-b", "W-b")]
CONN = {c: [] for c in COMBOS}     # (session, pre index, post index, 'E' or 'I')
CONN1 = []                          # variant: the spikes of one neuron jittered (Supplementary Fig. 9A), B-a, W-a
NPAIRS = 0
t1 = time.time()
for si, (s, x) in enumerate(S.items()):
    n = len(x.cells)
    for q, (a, b) in enumerate(itertools.combinations(range(n), 2)):
        e, C = fj.jitter_pair(x.times[a], x.times[b], M=1000, binw=0.0005, seed=si * 100000 + q)
        C1 = C.reshape(C.shape[0], -1, 2).sum(axis=2)
        e1 = e[::2]
        for bw, CC, ee in (("B-a", C1, e1), ("B-b", C, e)):
            for wd in ("W-a", "W-b"):
                r = fj.classify_ccg(ee, CC, width=wd)
                for d, (pre, post) in (("ab", (a, b)), ("ba", (b, a))):
                    exc, inh = r[d]
                    if exc:
                        CONN[(bw, wd)].append((s, pre, post, "E"))
                    if inh:
                        CONN[(bw, wd)].append((s, pre, post, "I"))
        # one-cell jitter: moving the spikes of a only tests b relative to a in both directions
        e1j, Cj = fj.jitter_pair(x.times[a], x.times[b], M=1000, binw=0.001, seed=si * 100000 + q + 50000, jitter_b=False)
        r = fj.classify_ccg(e1j, Cj, width="W-a")
        for d, (pre, post) in (("ab", (a, b)), ("ba", (b, a))):
            exc, inh = r[d]
            if exc:
                CONN1.append((s, pre, post, "E"))
            if inh:
                CONN1.append((s, pre, post, "I"))
    NPAIRS += n * (n - 1)
print(f"directed pairs {NPAIRS}; jitter tests took {time.time() - t1:.0f} s;",
      "peak memory %.1f GB" % (resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6))

In [ ]:
def cell_types(conns):
    """Putative pyramidal: a significant short-latency peak as reference; interneuron: a trough (Figure 3a legend).
    Returns dict (session, unit) -> 'pyr', 'int', 'both' or 'un'."""
    E = {(s, a) for s, a, b, k in conns if k == "E"}
    I = {(s, a) for s, a, b, k in conns if k == "I"}
    out = {}
    for s, x in S.items():
        for u in range(len(x.cells)):
            k = (s, u)
            out[k] = "both" if (k in E and k in I) else "pyr" if k in E else "int" if k in I else "un"
    return out
TYPES = {c: cell_types(v) for c, v in CONN.items()}
MAIN = ("B-a", "W-a")
TY = TYPES[MAIN]
for c in COMBOS:
    print(c, "types:", dict(collections.Counter(TYPES[c].values())), "; connections:",
          dict(collections.Counter(k for *_, k in CONN[c])))
# agreement with the lab's cell types (CellsInfo codes 1 pyr, 2 int, 3 unknown)
lab_map = {1: "pyr", 2: "int", 3: "un"}
ct = collections.Counter((lab_map[S[s].lab_type[u]], TY[(s, u)]) for (s, u) in TY)
print("lab type vs type found here (B-a, W-a):", dict(ct))

## Claim fig3-connection-rates

> Of 62,408 directed pairs of simultaneously recorded neurons, 495 (0.79%) show a short-latency narrow peak or trough in the cross-correlogram by the jitter test, 0.55% excitatory and 0.24% inhibitory, and 0.17% of pairs are reciprocally connected.

Reciprocal: a pair connected in both directions (any type), over (a) directed pairs and (b) unordered pairs.

In [ ]:
RATES = {}
for c in COMBOS:
    v = CONN[c]
    dirs = {(s, a, b) for s, a, b, k in v}
    nE = sum(k == "E" for *_, k in v)
    nI = sum(k == "I" for *_, k in v)
    recip = {(s, min(a, b), max(a, b)) for s, a, b in dirs if (s, b, a) in dirs}
    tot = pct(len(dirs), NPAIRS)
    RATES[c] = (tot, pct(nE, NPAIRS), pct(nI, NPAIRS), pct(len(recip), NPAIRS), pct(len(recip), NPAIRS / 2))
    r = RATES[c]
    print(f"{c}: directed pairs {NPAIRS} (paper 62,408); connected {len(dirs)} = {r[0]:.3f}% (0.79 +/- 0.079: {abs(r[0]-0.79)<=0.079}); "
          f"excitatory {nE} = {r[1]:.3f}% (0.55 +/- 0.055: {abs(r[1]-0.55)<=0.055}); inhibitory {nI} = {r[2]:.3f}% (0.24 +/- 0.024: {abs(r[2]-0.24)<=0.024}); "
          f"reciprocal pairs {len(recip)} = {r[3]:.3f}% of directed (a), {r[4]:.3f}% of unordered (b) (0.17 +/- 0.017: {abs(r[3]-0.17)<=0.017} / {abs(r[4]-0.17)<=0.017})")
print("per session (B-a, W-a), % of directed pairs connected:",
      {s: round(pct(sum(1 for r in CONN[MAIN] if r[0] == s), len(x.cells) * (len(x.cells) - 1)), 2) for s, x in S.items()})
v = CONN1
dirs1 = {(s, a, b) for s, a, b, k in v}
nE1, nI1 = sum(k == "E" for *_, k in v), sum(k == "I" for *_, k in v)
rec1 = {(s, min(a, b), max(a, b)) for s, a, b in dirs1 if (s, b, a) in dirs1}
print(f"variant, one-cell jitter (Supplementary Fig. 9A), B-a W-a: connected {len(dirs1)} = {pct(len(dirs1), NPAIRS):.3f}%, excitatory {nE1} = {pct(nE1, NPAIRS):.3f}%, "
      f"inhibitory {nI1} = {pct(nI1, NPAIRS):.3f}%, reciprocal {len(rec1)} = {pct(len(rec1), NPAIRS):.3f}% of directed; "
      f"shared with the two-cell jitter: {len(dirs1 & {(s, a, b) for s, a, b, k in CONN[MAIN]})}")

**Result: not reproduced.** Over 40,240 directed pairs (three rats; the paper's 62,408 are from four), 1.07% to 1.45% are connected, 0.90% to 1.17% excitatory and 0.18% to 0.28% inhibitory, depending on the bin and width readings; none of the three directed percentages is within its tolerance under any combination. The reciprocal fraction is within tolerance under (B-a, W-b) over directed pairs (0.162%) and (B-b, W-b) over unordered pairs (0.179%). Jittering the spikes of one neuron only (Supplementary Figure 9A) finds more connections, not fewer (1.521% connected, 1.285% excitatory), so it does not explain the excess over the paper.

## Claim fig3-cell-classes

> By their cross-correlogram interactions 32.5% of units are putative pyramidal cells and 12.5% putative interneurons; 50.7% of the targets of excitatory cells themselves suppress other neurons; the ratio of putative pyramidal cells to interneurons is 2.82 +/- 0.51 (s.d.).

Readings C-a (a unit with both a peak and a trough is unclassified) and C-b (it is left out of the denominators); ratio readings (a) per session (sessions with at least one interneuron), (b) per rat, (c) pooled.

In [ ]:
def classes(c, rd):
    T = TYPES[c]
    keys = [k for k in T if not (rd == "C-b" and T[k] == "both")]
    npyr = sum(T[k] == "pyr" for k in keys)
    nint = sum(T[k] == "int" for k in keys)
    targets = {(s, b) for s, a, b, k in CONN[c] if k == "E"}
    targets = [t for t in targets if not (rd == "C-b" and T[t] == "both")]
    supp = sum(T[t] in ("int",) or (rd == "C-a" and T[t] == "both" and False) for t in targets)
    # a target 'suppresses other neurons' when it has a significant trough as reference
    supp = sum(t in {(s, a) for s, a, b, k in CONN[c] if k == "I"} for t in targets)
    def ratio(group):
        vals = []
        for gname in sorted({group(k) for k in keys}):
            ks = [k for k in keys if group(k) == gname]
            p_, i_ = sum(T[k] == "pyr" for k in ks), sum(T[k] == "int" for k in ks)
            if i_ > 0:
                vals.append(p_ / i_)
        return np.mean(vals), np.std(vals, ddof=1) if len(vals) > 1 else float("nan"), len(vals)
    return dict(pyr=pct(npyr, len(keys)), int=pct(nint, len(keys)), supp=pct(supp, len(targets)),
                n=len(keys), ntar=len(targets), r_ses=ratio(lambda k: k[0]), r_rat=ratio(lambda k: k[0][:2]),
                r_pool=npyr / nint if nint else float("nan"))
CLS = {}
for c in COMBOS:
    for rd in ("C-a", "C-b"):
        r = classes(c, rd)
        CLS[(c, rd)] = r
        ok = [abs(r["pyr"] - 32.5) <= 3.25, abs(r["int"] - 12.5) <= 1.25, abs(r["supp"] - 50.7) <= 5.07]
        rok = [abs(r["r_ses"][0] - 2.82) <= 0.51, abs(r["r_rat"][0] - 2.82) <= 0.51, abs(r["r_pool"] - 2.82) <= 0.51]
        print(f"{c} {rd}: pyramidal {r['pyr']:.1f}%, interneurons {r['int']:.1f}% of {r['n']} units; "
              f"targets of excitatory cells with a trough {r['supp']:.1f}% of {r['ntar']}; "
              f"ratio per session {r['r_ses'][0]:.2f} +/- {r['r_ses'][1]:.2f} (n={r['r_ses'][2]}), per rat {r['r_rat'][0]:.2f} +/- {r['r_rat'][1]:.2f} (n={r['r_rat'][2]}), pooled {r['r_pool']:.2f}; "
              f"within tolerance: pyr {ok[0]}, int {ok[1]}, 50.7% {ok[2]}, ratio (a/b/c) {rok}")
print("Figure 2d fractions for comparison: 34.4% and 13.3%")
# Troughs next to peaks (verifier, round 1): with a strong peak the jittered surrogates are raised over the whole window,
# so a trough elsewhere in the same CCG may come from the peak. How many inhibitory connections share their pair with an
# excitatory one?
for c in (MAIN, ("B-b", "W-b")):
    E = {(s, a, b) for s, a, b, k in CONN[c] if k == "E"}
    I = [(s, a, b) for s, a, b, k in CONN[c] if k == "I"]
    same = sum((s, a, b) in E for s, a, b in I); rev = sum((s, b, a) in E for s, a, b in I)
    alone = sum((s, a, b) not in E and (s, b, a) not in E for s, a, b in I)
    both_u = [k for k, t in TYPES[c].items() if t == "both"]
    ref_I = {(s, a) for s, a, b in I if (s, a, b) not in E and (s, b, a) not in E}
    print(f"{c}: {len(I)} inhibitory connections, {same} in a pair with an excitatory one in the same direction, {rev} in the reverse direction, "
          f"{alone} in pairs with no peak; units with both a peak and a trough {len(both_u)}, of which {sum(k in ref_I for k in both_u)} keep a trough in a pair without a peak")
    D = {(s, a, b) for s, a, b, k in CONN[c]}
    rec = {(s, min(a, b), max(a, b)) for s, a, b in D if (s, b, a) in D}
    kinds = collections.Counter(tuple(sorted(k for s_, a_, b_, k in CONN[c] if s_ == s and {a_, b_} == {a, b})) for s, a, b in rec)
    print(f"   reciprocal pairs {len(rec)}, by the types of their connections: {dict(kinds)}")
# a few such CCGs: real counts, surrogate mean and the global band, in 1 ms bins over -5 to +5 ms
E = {(s, a, b) for s, a, b, k in CONN[MAIN] if k == "E"}
ex_I = [(s, a, b) for s, a, b, k in CONN[MAIN] if k == "I" and (s, b, a) in E][:3]
for s, a, b in ex_I:
    e_, C_ = fj.jitter_pair(S[s].times[a], S[s].times[b], M=1000, binw=0.001, seed=7)
    sur = C_[1:]
    print(f"   {s} {S[s].cells[a]} -> {S[s].cells[b]} (trough; peak from {S[s].cells[b]} -> {S[s].cells[a]}): real {C_[0].tolist()}; "
          f"surrogate mean {np.round(sur.mean(0), 1).tolist()}; global band [{np.quantile(sur.min(1), 0.01):.0f}, {np.quantile(sur.max(1), 0.99):.0f}]")

**Result: not reproduced.** Putative pyramidal cells are 37.5% to 45.1% of units and putative interneurons 2.6% to 4.4% under every combination, the share of excitatory targets with a trough is 9.4% to 25.5%, and the pyramidal to interneuron ratio is 9.9 to 14.8 per session, rat or pooled; none of the four values is within tolerance. Many units have both a peak and a trough (26 to 39), and few have a trough alone. Of the 112 inhibitory connections (B-a, W-a), 58 lie in a pair with an excitatory peak in the reverse direction, and 57 of the 80 reciprocal pairs are such an excitatory and inhibitory pair; in the CCGs printed the surrogate mean is raised over the window by the jittered peak (EE.152 84 -> 78: surrogate means 34.0 to 38.6 in the 1 to 4 ms bins against real counts of 16 to 24, beside a peak of 194 at -2 to -1 ms), so many of these troughs may come from the peak. Leaving them out would lower the interneuron count further, away from the paper's 12.5%.

## Claim pyr-pyr-connections

> Monosynaptic excitation between putative pyramidal cells is rare, found in 0.12% of layer 2/3 and 0.27% of layer 5 pairs.

Excitatory connections whose target is also a putative pyramidal cell (B-a, W-a; C-a), by layer under each rat reading; denominators (a) all directed pairs of the layer, (b) directed pairs of two putative pyramidal cells.

In [ ]:
def npairs_rat(pred):
    return sum(len(x.cells) * (len(x.cells) - 1) for s, x in S.items() if pred(s))
for c in (MAIN, ("B-b", "W-b")):
    T = TYPES[c]
    pp = [(s, a, b) for s, a, b, k in CONN[c] if k == "E" and T[(s, b)] == "pyr"]
    for name, l5 in LAYER_READINGS.items():
        out = []
        for layer in ("L2/3", "L5"):
            inl = lambda s: layer_of(s[:2], l5) == layer
            n_all = npairs_rat(inl)
            npyr = {s: sum(T[(s, u)] == "pyr" for u in range(len(x.cells))) for s, x in S.items()}
            n_pp = sum(v * (v - 1) for s, v in npyr.items() if inl(s))
            k = sum(inl(s) for s, a, b in pp)
            out.append((layer, k, pct(k, n_all), pct(k, n_pp), n_all, n_pp))
        print(c, name, "; ".join(f"{l}: {k} connections, (a) {a:.3f}% of {na}, (b) {b:.3f}% of {nb}" for l, k, a, b, na, nb in out),
              "| within tolerance (a):", abs(out[0][2]-0.12) <= 0.012 and abs(out[1][2]-0.27) <= 0.027,
              "(b):", abs(out[0][3]-0.12) <= 0.012 and abs(out[1][3]-0.27) <= 0.027,
              "| layer 5 above layer 2/3 (a):", out[1][2] > out[0][2], "(b):", out[1][3] > out[0][3])

**Result: partially reproduced.** Under R-a, layer 5 is above layer 2/3 by both denominators (B-a, W-a: 0.355% against 0.261% of all directed pairs; 1.906% against 1.146% of pyramidal pairs), but no value is within 0.12 +/- 0.012 or 0.27 +/- 0.027.

## Claim nearby-pyr-int-probability

> The connection probability between nearby (under 200 um) putative pyramidal cells and interneurons is 1 to 2%.

Same shank; readings (a) pyramidal to interneuron excitatory connections over such directed pairs, (b) a connection either way over unordered pairs. Range 0.9 to 2.2%.

In [ ]:
for c in COMBOS:
    T = TYPES[c]
    dirsE = {(s, a, b) for s, a, b, k in CONN[c] if k == "E"}
    dirs = {(s, a, b) for s, a, b, k in CONN[c]}
    nd = ne = nu = nc = 0
    for s, x in S.items():
        P_ = [u for u in range(len(x.cells)) if T[(s, u)] == "pyr"]
        I_ = [u for u in range(len(x.cells)) if T[(s, u)] == "int"]
        for a in P_:
            for b in I_:
                if x.shank[a] == x.shank[b]:
                    nd += 1
                    ne += (s, a, b) in dirsE
                    nu += 1
                    nc += ((s, a, b) in dirs) or ((s, b, a) in dirs)
    print(f"{c}: (a) {ne}/{nd} = {pct(ne, nd):.2f}% ; (b) {nc}/{nu} = {pct(nc, nu):.2f}% ; in 0.9-2.2%: {0.9 <= pct(ne, nd) <= 2.2} / {0.9 <= pct(nc, nu) <= 2.2}")

**Result: not reproduced.** With cell types from the jitter test, 45% to 59% of same-shank pyramidal and interneuron pairs are connected under every combination and both direction readings, far above 0.9 to 2.2%; the interneurons found here are few (16 to 26 units) and are defined by their own connections. (The count of excitatory connections in reading (a) now uses the set of excitatory connections directly; the values are unchanged.)

## Claim fig3b-session-connections

> In the single layer 2/3 session of Figure 3b (117 neurons, 117 squared pairs) 48 connections have a putative pyramidal reference cell and 30 a putative interneuron; in the layer 5 session of Supplementary Figure 11 (142 neurons) the numbers are 73 and 25.

The layer 2/3 session is EE.188 (117 cells, named by the CRCNS description). No session holds 142 cells (largest FF session: FF.130 with 64), so the layer 5 session cannot be identified. Readings (a) directed connections, (b) distinct reference cells. The lab's own list of connections in EE188_example.mat (pairs_all) is compared.

In [ ]:
x = S["EE188"]
print("EE.188 cells", len(x.cells), "; ordered pairs", len(x.cells) * (len(x.cells) - 1), "(paper 13,572)")
print("sessions by cell count:", {s: len(v.cells) for s, v in S.items()}, "; a session of 142 cells:", any(len(v.cells) == 142 for v in S.values()))
for c in COMBOS:
    v = [r for r in CONN[c] if r[0] == "EE188"]
    T = TYPES[c]
    nE = sum(k == "E" for *_, k in v); nI = sum(k == "I" for *_, k in v)
    dE = len({a for s, a, b, k in v if k == "E"}); dI = len({a for s, a, b, k in v if k == "I"})
    print(f"{c}: (a) directed connections, pyramidal reference {nE}, interneuron reference {nI}; "
          f"(b) distinct reference cells {dE} and {dI}; expected 48 +/- 4.8 and 30 +/- 3: "
          f"(a) {abs(nE-48)<=4.8 and abs(nI-30)<=3}, (b) {abs(dE-48)<=4.8 and abs(dI-30)<=3}")
pa = ex["pairs_all"].astype(int)
CIDX = {c: i for i, c in enumerate(x.cells)}
lab_pairs = {(CIDX[a], CIDX[b]) for a, b in pa}
for c in COMBOS:
    mine = {(a, b) for s, a, b, k in CONN[c] if s == "EE188"}
    mineE = {(a, b) for s, a, b, k in CONN[c] if s == "EE188" and k == "E"}
    print(f"{c}: lab pairs_all {len(lab_pairs)}; found here among them {len(lab_pairs & mine)} "
          f"(as excitatory {len(lab_pairs & mineE)}); connections here not in the lab list {len(mine - lab_pairs)}")
mine1 = {(a, b) for s, a, b, k in CONN1 if s == "EE188"}
v1 = [r for r in CONN1 if r[0] == "EE188"]
print(f"variant, one-cell jitter (B-a, W-a): EE.188 directed connections, pyramidal reference {sum(k == 'E' for *_, k in v1)}, interneuron reference {sum(k == 'I' for *_, k in v1)}; "
      f"lab pairs_all found {len(lab_pairs & mine1)} of {len(lab_pairs)}; not in the lab list {len(mine1 - lab_pairs)}")

**Result: not reproduced.** No session holds 142 cells in the lab's lists or among the NWB mPFC units (largest FF session 64 and 68), so the layer 5 session is not in the datasets. In EE.188 (117 cells, 13,572 ordered pairs, as the paper), the connections with a pyramidal reference cell number 81 to 119 and those with an interneuron reference 16 to 22 (directed), or 59 to 67 and 9 to 16 distinct reference cells, against 48 and 30. Of the lab's 43 listed connections (pairs_all), 35 to 40 are found here, all as excitatory. With the one-cell jitter, 137 and 15 directed connections.

## Claim fig3d-hub

> A large part of the recorded neurons belong to a single interconnected circuit, 39 of 117 (33%) in the layer 2/3 session and 45 of 142 in the layer 5 session.

Largest connected component of the undirected graph of excitatory and inhibitory connections in EE.188. The layer 5 session is not identified (above); the largest component of FF.130, the largest FF session, is shown as a stand-in.

In [ ]:
def largest_component(edges, n):
    parent = list(range(n))
    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i
    for a, b in edges:
        parent[find(a)] = find(b)
    roots = collections.Counter(find(i) for i in range(n) if any(i in e for e in edges))
    return max(roots.values()) if roots else 0
for c in COMBOS:
    out = []
    for s in ("EE188", "FF130"):
        e = [(a, b) for ss, a, b, k in CONN[c] if ss == s]
        out.append((s, largest_component(e, len(S[s].cells)), len(S[s].cells)))
    print(c, "; ".join(f"{s}: {k} of {n}" for s, k, n in out), "| EE188 within 39 +/- 3.9:", abs(out[0][1] - 39) <= 3.9)
e = [(CIDX[a], CIDX[b]) for a, b in pa]
print("lab pairs_all graph, EE188: largest component", largest_component(e, len(x.cells)))
print("variant, one-cell jitter (B-a, W-a): EE188 largest component", largest_component([(a, b) for ss, a, b, k in CONN1 if ss == "EE188"], len(x.cells)), "of", len(x.cells))

**Result: not reproduced.** The largest connected circuit in EE.188 holds 65 to 86 of 117 cells with the connections found here (93 with the one-cell jitter), outside 39 +/- 3.9; the layer 5 session is not in the datasets. The lab's own connection list (pairs_all) gives a largest circuit of exactly 39 cells.

## Claim fig3f-distance

> Most connected pairs are on the same shank and connection probability falls rapidly with the distance between the neurons, yet connections are found up to 1,200 um apart.

Probability by shank separation (200 um per shank), excitatory and inhibitory apart, with exact Clopper-Pearson 99% intervals.

In [ ]:
from scipy.stats import binomtest
for c in (MAIN, ("B-b", "W-b")):
    pairs_by = collections.Counter()
    for s, v in S.items():
        for a in range(len(v.cells)):
            for b in range(len(v.cells)):
                if a != b:
                    pairs_by[abs(int(v.shank[a]) - int(v.shank[b]))] += 1
    conn_by = {k: collections.Counter() for k in "EI"}
    for s, a, b, k in CONN[c]:
        conn_by[k][abs(int(S[s].shank[a]) - int(S[s].shank[b]))] += 1
    print(c)
    for k in "EI":
        row = []
        for dsh in range(8):
            n, m = pairs_by[dsh], conn_by[k][dsh]
            ci = binomtest(m, n).proportion_ci(confidence_level=0.99, method="exact") if n else None
            row.append((dsh * 200, m, n, pct(m, n), ci))
        print("  ", k, "; ".join(f"{d} um: {m}/{n} = {p:.2f}% [{ci.low*100:.2f}, {ci.high*100:.2f}]" for d, m, n, p, ci in row if n))
    allc = conn_by["E"] + conn_by["I"]
    same, other = allc[0], sum(v for d, v in allc.items() if d > 0)
    ci0 = binomtest(allc[0], pairs_by[0]).proportion_ci(0.99, "exact")
    ci1 = binomtest(allc[1], pairs_by[1]).proportion_ci(0.99, "exact")
    maxd = max(d for d, v in allc.items() if v > 0) * 200
    print(f"  all connections: same shank {same}, other shanks {other}; same-shank {pct(allc[0], pairs_by[0]):.2f}% [{ci0.low*100:.2f}, {ci0.high*100:.2f}] "
          f"vs 200 um {pct(allc[1], pairs_by[1]):.2f}% [{ci1.low*100:.2f}, {ci1.high*100:.2f}], intervals separate: {ci0.low > ci1.high}; largest separation with a connection {maxd} um")

**Result: partially reproduced.** More connected pairs are on the same shank than on all others together (384 against 200, B-a W-a; 278 against 155, B-b W-b), and the same-shank probability (6.77% [5.93, 7.67]) is above the 200 um probability (1.29% [1.01, 1.62]) with separate 99% intervals; but connections are found up to 1,400 um, not 1,200 um.

## Claim fig3-trajectory-by-class

> Large fractions of putative pyramidal cells (72.2% in layer 2/3, 84.7% in layer 5) and of putative interneurons (58.3% and 83%) are trajectory specific.

A cell is trajectory specific when the left-right test gives a significant segment anywhere. Types from B-a, W-a (C-a). Readings (a) the Figure 3 sessions (EE.188 for layer 2/3; the layer 5 session is not identified, FF.130 shown as a stand-in), (b) all sessions of each layer under each rat reading.

In [ ]:
def traj_frac(keys, typ, T=None):
    T = T or TY
    ks = [k for k in keys if T[k] == typ]
    h = sum(SEL[k]["significant"] for k in ks)
    return pct(h, len(ks)), h, len(ks)
for lab_, sess in (("(a) EE188 / FF130 stand-in", ("EE188", "FF130")),):
    for typ in ("pyr", "int"):
        a = traj_frac([k for k in KEYS if k[0] == sess[0]], typ); b = traj_frac([k for k in KEYS if k[0] == sess[1]], typ)
        print(f"{lab_} {typ}: EE188 {a[0]:.1f}% ({a[1]}/{a[2]}), FF130 {b[0]:.1f}% ({b[1]}/{b[2]})")
for name, l5 in LAYER_READINGS.items():
    vals = {}
    for layer in ("L2/3", "L5"):
        for typ in ("pyr", "int"):
            vals[(layer, typ)] = traj_frac(keys_layer(layer, l5), typ)
    ok = [abs(vals[("L2/3", "pyr")][0] - 72.2) <= 7.22, abs(vals[("L5", "pyr")][0] - 84.7) <= 8.47,
          abs(vals[("L2/3", "int")][0] - 58.3) <= 5.83, abs(vals[("L5", "int")][0] - 83) <= 8.3]
    print(f"(b) {name}: " + "; ".join(f"{l} {t} {v[0]:.1f}% ({v[1]}/{v[2]})" for (l, t), v in vals.items()), "| within tolerance:", ok)
print("all neurons, any significant segment:", f"{pct(sum(SEL[k]['significant'] for k in KEYS), len(KEYS)):.1f}%",
      "; null relabelling", f"{pct(sum(NULL1[k]['significant'] for k in KEYS), len(KEYS)):.1f}%")

**Result: partially reproduced.** Under R-a, all sessions, 66.8% of layer 2/3 pyramidal cells (141/211) and 52.6% of layer 2/3 interneurons (10/19) are trajectory specific, within tolerance; layer 5 pyramidal cells (72.2%, 39/54) and interneurons (50.0%, 2/4) are not. In EE.188 the fractions are 67.2% (39/58) and 42.9% (3/7).

## Claim no-spatial-clustering

> Left- and right-specific neurons are not clustered on the probe but spread over the whole recorded volume, and neurons active at any part of the maze are found on all shanks.

Chi-square tests in the Figure 3 session (EE.188; the layer 5 session is not identified): (i) shank against class (left, right, both, none; left is a segment where left exceeds right); (ii) shank against the tenth of the maze holding the neuron's peak rate (PSTH of the side with the higher peak), with the peak taken from the rate under each of the claims file's readings P-a (occupancy normalised) and P-b (the Methods' PSTH). Cramer's V; a permutation p-value (shank labels shuffled 2,000 times) is shown with the asymptotic one, as many cells are small. All sessions as a variant.

In [ ]:
def peak_tenth(k, pr="P-b"):
    """Tenth of the maze holding the neuron's peak, on the side with the higher peak; rate readings P-a (occupancy
    normalised, Hz) and P-b (the Methods' PSTH), as the claims file header asks wherever peaks matter."""
    s, u = k
    P = PSTH[k]
    lab = S[s].side == "L"
    if pr == "P-a":
        O = OCC[s]
        with np.errstate(invalid="ignore", divide="ignore"):
            pl, pr_ = np.nan_to_num(P[lab].sum(0) / O[lab].sum(0)), np.nan_to_num(P[~lab].sum(0) / O[~lab].sum(0))
    else:
        pl, pr_ = P[lab].mean(0), P[~lab].mean(0)
    v = pl if pl.max() >= pr_.max() else pr_
    return min(int(fj.GRID[np.argmax(v)] * 10), 9)
def sel_class(k):
    sg = {g[2] for g in SEL[k]["segments"]}
    return "both" if sg == {1, -1} else "left" if sg == {1} else "right" if sg == {-1} else "none"
def chi_test(keys, f, rng):
    sh = np.array([S[k[0]].shank[k[1]] for k in keys])
    cl = np.array([f(k) for k in keys])
    tab = np.array([[np.sum((sh == a) & (cl == b)) for b in np.unique(cl)] for a in np.unique(sh)])
    chi2, p, dof, _ = st.chi2_contingency(tab)
    V = np.sqrt(chi2 / (tab.sum() * (min(tab.shape) - 1)))
    null = []
    for _ in range(2000):
        shp = rng.permutation(sh)
        t2 = np.array([[np.sum((shp == a) & (cl == b)) for b in np.unique(cl)] for a in np.unique(sh)])
        null.append(st.chi2_contingency(t2)[0])
    pperm = (np.sum(np.array(null) >= chi2) + 1) / (len(null) + 1)
    return chi2, p, pperm, V, tab
rng = np.random.default_rng(7)
for s in ("EE188",):
    ks = [k for k in KEYS if k[0] == s]
    for nm, f in (("(i) shank x class", sel_class), ("(ii) shank x peak tenth, P-b", peak_tenth),
                  ("(ii) shank x peak tenth, P-a", lambda k: peak_tenth(k, "P-a"))):
        chi2, p, pp, V, tab = chi_test(ks, f, rng)
        print(f"{s} {nm}: chi2 {chi2:.1f}, asymptotic p {p:.3f}, permutation p {pp:.3f}, Cramer's V {V:.2f}; table (shanks x classes):\n{tab}")
print("variant, every session:")
for s in S:
    ks = [k for k in KEYS if k[0] == s]
    if len(ks) < 15:
        continue
    r1 = chi_test(ks, sel_class, np.random.default_rng(8))
    r2 = chi_test(ks, peak_tenth, np.random.default_rng(9))
    r3 = chi_test(ks, lambda k: peak_tenth(k, "P-a"), np.random.default_rng(10))
    print(f"  {s}: (i) perm p {r1[2]:.3f} V {r1[3]:.2f}; (ii) P-b perm p {r2[2]:.3f} V {r2[3]:.2f}; (ii) P-a perm p {r3[2]:.3f} V {r3[3]:.2f}")

**Result: reproduced (layer 2/3 session only).** In EE.188 neither shank against class (chi-square 24.4, permutation p = 0.265, Cramer's V 0.26) nor shank against peak tenth shows an association at p < 0.05, with peaks from either rate reading (P-b permutation p = 0.232, V 0.28; P-a p = 0.173, V 0.30). The layer 5 session of Supplementary Figure 11B is not in the datasets, so it was not tested. Across the ten sessions with 15 or more cells, two of the 20 permutation tests of shank against class and shank against P-b peak tenth fall below 0.05 (EE.210 shank against class, p = 0.022; FF.103 shank against P-b peak tenth, p = 0.013), and none of the ten P-a peak tests does.

## Claim fig2-population-uniform

> Individual neurons fire transiently at specific locations, but the population as a whole fires relatively uniformly over the whole maze.

In each Figure 2a session (EE.188; the layer 5 session is not identified, FF.130 shown as a stand-in): the position of each neuron's peak (side with the higher peak; rate readings P-a and P-b of the claims file header), counted per tenth of the maze, against no tenth above twice or below half its share; and the population mean rate per tenth (spikes over occupancy time, Hz) within 50% of its overall mean. Chi-square of peak positions against uniform, and all sessions, as variants.

In [ ]:
def pop_uniform(s, pr="P-b"):
    ks = [k for k in KEYS if k[0] == s]
    cnt = np.bincount([peak_tenth(k, pr) for k in ks], minlength=10)
    exp = len(ks) / 10
    x = S[s]
    occ = OCC[s].sum(0)
    rates = np.array([PSTH[k].sum(0) / occ for k in ks]).mean(0)   # Hz on the grid
    tenth = np.minimum((fj.GRID * 10).astype(int), 9)
    rt = np.array([rates[tenth == i].mean() for i in range(10)])
    ok1 = bool(np.all(cnt <= 2 * exp) and np.all(cnt >= 0.5 * exp))
    ok2 = bool(np.all(np.abs(rt - rt.mean()) <= 0.5 * rt.mean()))
    p = st.chisquare(cnt).pvalue
    return cnt, exp, rt, ok1, ok2, p
for pr in ("P-b", "P-a"):
    print(f"peaks from {pr}:")
    for s in S:
        cnt, exp, rt, ok1, ok2, p = pop_uniform(s, pr)
        tag = " (Figure 2a layer 2/3 session)" if s == "EE188" else " (stand-in for the layer 5 session)" if s == "FF130" else ""
        print(f"  {s}{tag}: peaks per tenth {cnt.tolist()} (expected {exp:.1f}; within x0.5-x2: {ok1}; chi-square p {p:.3f})"
              + (f"; population rate per tenth {np.round(rt, 2).tolist()} Hz (range {rt.min():.2f}-{rt.max():.2f}; within 50% of mean {rt.mean():.2f}: {ok2})" if pr == "P-b" else ""))

**Result: not reproduced.** In EE.188 the peaks are not spread over the maze under either rate reading: with the Methods' PSTH (P-b) 24 and 59 of 117 neurons peak in the first and last tenth, and with the occupancy-normalised rate (P-a) 46 peak in the first tenth, against 11.7 expected; under P-b several tenths hold one or none, under P-a the tenths hold 1 to 46 (chi-square p < 0.001 under both). Every session fails the peak criterion under both readings. The population rate per tenth stays within 50% of its mean in EE.188 (1.84 to 2.85 Hz, mean 2.30) and in 11 of 13 sessions.

## Claim fig2c-active-fraction

> About 10% of layer 2/3 neurons and 20% of layer 5 neurons fire at least one spike in a given 100 ms window, a fraction relatively constant over the maze.

Consecutive 100 ms windows from each trial's start to its end; the percentage of the session's neurons with at least one spike, averaged over windows; per tenth of the maze by the position at each window's centre. Readings (a) the Figure 2a sessions (EE.188; FF.130 as stand-in for the layer 5 session), (b) all sessions of each layer, windows pooled, under each rat reading. Every NWB mPFC unit as the session's neurons is shown as a unit-list variant.

In [ ]:
ACT = {}
for s, x in S.items():
    vals, pos = [], []
    for a, b in zip(x.start, x.stop):
        edges = np.arange(a, b - 0.1 + 1e-9, 0.1)
        if len(edges) == 0:
            continue
        cnt = np.zeros(len(edges))
        for t in x.times:
            h = np.histogram(t, bins=np.r_[edges, edges[-1] + 0.1])[0]
            cnt += h > 0
        vals.append(100 * cnt / len(x.cells))
        pos.append(x.position(edges + 0.05))
    ACT[s] = (np.concatenate(vals), np.concatenate(pos))
def act_summary(sessions):
    v = np.concatenate([ACT[s][0] for s in sessions]); p = np.concatenate([ACT[s][1] for s in sessions])
    ok = np.isfinite(p)
    tenth = np.minimum((p[ok] * 10).astype(int), 9)
    mt = np.array([v[ok][tenth == i].mean() for i in range(10)])
    return v.mean(), v.std(), mt, bool(np.all(np.abs(mt - v.mean()) <= 0.5 * v.mean()))
for s in S:
    m, sd, mt, ok = act_summary([s])
    print(f"{s}: {m:.1f} +/- {sd:.1f}% active per 100 ms; per tenth {np.round(mt,1).tolist()}; constant within 50%: {ok}")
m23 = act_summary(["EE188"]); m5 = act_summary(["FF130"])
print(f"(a) EE188 {m23[0]:.1f}% (10 +/- 1: {abs(m23[0]-10)<=1}); FF130 stand-in {m5[0]:.1f}% (20 +/- 2: {abs(m5[0]-20)<=2})")
for name, l5 in LAYER_READINGS.items():
    a = act_summary([s for s in S if layer_of(s[:2], l5) == "L2/3"]); b = act_summary([s for s in S if layer_of(s[:2], l5) == "L5"])
    print(f"(b) {name}: layer 2/3 {a[0]:.1f}% (10 +/- 1: {abs(a[0]-10)<=1}; constant {a[3]}), layer 5 {b[0]:.1f}% (20 +/- 2: {abs(b[0]-20)<=2}; constant {b[3]})")
# variant on the unit list: every NWB mPFC unit of the session (data check)
ACT_ALL = {}
for s, x in S.items():
    vals, pos = [], []
    for a, b in zip(x.start, x.stop):
        edges = np.arange(a, b - 0.1 + 1e-9, 0.1)
        if len(edges) == 0:
            continue
        cnt = np.zeros(len(edges))
        for t in NWBU[s]["times"]:
            cnt += np.histogram(t, bins=np.r_[edges, edges[-1] + 0.1])[0] > 0
        vals.append(100 * cnt / len(NWBU[s]["times"]))
        pos.append(x.position(edges + 0.05))
    ACT_ALL[s] = (np.concatenate(vals), np.concatenate(pos))
ACT_LAB, ACT = ACT, ACT_ALL
m23 = act_summary(["EE188"]); m5 = act_summary(["FF130"])
print(f"variant, every NWB mPFC unit: (a) EE188 {m23[0]:.1f}% of {len(NWBU['EE188']['times'])} (10 +/- 1: {abs(m23[0]-10)<=1}); FF130 {m5[0]:.1f}% (20 +/- 2: {abs(m5[0]-20)<=2})")
for name, l5 in LAYER_READINGS.items():
    a = act_summary([s for s in S if layer_of(s[:2], l5) == "L2/3"]); b = act_summary([s for s in S if layer_of(s[:2], l5) == "L5"])
    print(f"   (b) {name}: layer 2/3 {a[0]:.1f}% (10 +/- 1: {abs(a[0]-10)<=1}; constant {a[3]}), layer 5 {b[0]:.1f}% (20 +/- 2: {abs(b[0]-20)<=2}; constant {b[3]})")
ACT = ACT_LAB

**Result: partially reproduced.** Under R-a the layer 5 sessions have 20.3% of neurons active per 100 ms (20 +/- 2), and the layer 2/3 sessions 12.6% (EE.188 alone 11.8%), outside 10 +/- 1; the fraction is constant within 50% over the tenths of the maze in both layers. Under R-b neither layer matches. The criterion names no partial outcome; one of its two values, the layer difference and the constancy hold. With every NWB mPFC unit the values fall (R-a: layer 2/3 6.7%, layer 5 18.6%; EE.188 6.5%), again with only one of the two within tolerance under any reading.

## Claim fig2-field-size

> Most neurons are active over a similar stretch of the maze, 0.27 (62 cm) +/- 0.17 (39 cm) of normalised length (mean +/- s.d.), measured between the 50% boundaries of the peak rate.

Width between the 50% boundaries of the PSTH's peak (Supplementary Figure 2B). Rate readings P-a (occupancy normalised, Hz) and P-b (the Methods' PSTH). Which PSTH: (a) all trials, (b) left and right as two entries, (c) the side with the higher peak. Width: (i) the contiguous run around the peak at or above 50%, (ii) the total length at or above 50%. Neurons: (x) the Figure 2a session (EE.188), (y) all units.

In [ ]:
dx = fj.GRID[1] - fj.GRID[0]
def curves(k, pr, which):
    s, u = k
    P = PSTH[k]; lab = S[s].side == "L"; O = OCC[s]
    def rate(m):
        return P[m].sum(0) / O[m].sum(0) if pr == "P-a" else P[m].mean(0)
    allm = np.ones(len(lab), bool)
    if which == "a":
        return [rate(allm)]
    l, r = rate(lab), rate(~lab)
    if which == "b":
        return [l, r]
    return [l if l.max() >= r.max() else r]
def width(v, how):
    if v.max() <= 0:
        return np.nan
    above = v >= 0.5 * v.max()
    if how == "ii":
        return above.sum() * dx
    p = int(np.argmax(v)); a = b = p
    while a > 0 and above[a - 1]:
        a -= 1
    while b < len(v) - 1 and above[b + 1]:
        b += 1
    return (b - a + 1) * dx
FS_RES = []
for pr in ("P-a", "P-b"):
    for which in "abc":
        for how in ("i", "ii"):
            for ns, keys in (("x", [k for k in KEYS if k[0] == "EE188"]), ("y", KEYS)):
                w = np.array([width(v, how) for k in keys for v in curves(k, pr, which)])
                w = w[np.isfinite(w)]
                m, sd = w.mean(), w.std(ddof=1)
                ok = abs(m - 0.27) <= 0.17 and abs(sd - 0.17) <= 0.017
                FS_RES.append((pr, which, how, ns, m, sd, len(w), ok))
                print(f"{pr} ({which}) ({how}) ({ns}): width {m:.3f} +/- {sd:.3f} (n={len(w)}); both within tolerance: {ok}")
print("combinations within tolerance:", [r[:4] for r in FS_RES if r[-1]])

**Result: reproduced.** Five combinations put both the mean and the s.d. within tolerance, for example the Methods' PSTH over all trials, total length above 50%, all units (0.286 +/- 0.185), and the occupancy-normalised rate of the side with the higher peak, contiguous run, EE.188 (0.240 +/- 0.177). Other combinations give means from 0.161 to 0.416.

## Claim population-vector-width

> Population vector correlations fall off over similar distances throughout the maze, 0.18 +/- 0.087 of normalised length in layer 2/3 and 0.27 +/- 0.13 in layer 5.

Population vectors of PSTHs (P-b, the Methods' PSTH; P-a as a variant) at each grid position. With raw rates the correlation between positions is dominated by the differences of mean rate between neurons, so each neuron's rate is also normalised, (peak) to its peak as Figure 2a displays the rates ("normalized relative to its peak firing rate"), or (z) z-scored over positions; this choice is not in the paper and the claims file, and the peak normalisation is adopted as the assumption, being the only normalisation the paper states for these rates; Pearson correlation between positions across neurons; width at each position: the length of the contiguous run around the diagonal where the correlation is at least half its diagonal value (1); mean +/- s.d. over positions. Readings (a) left and right PSTHs concatenated, (b) all trials. Sessions: EE.188 for layer 2/3 (the layer 5 session is not identified; FF.130 stand-in), all sessions of each layer as a variant.

In [ ]:
def pv_width(keys, rd, pr="P-b", norm="peak"):
    rows = []
    for k in keys:
        if rd == "a":
            v = np.vstack(curves(k, pr, "b"))
        else:
            v = curves(k, pr, "a")[0][None, :]
        v = np.nan_to_num(v)
        if norm == "peak":      # each neuron's rate relative to its peak, as Figure 2a displays it
            v = v / (v.max() if v.max() > 0 else 1)
        elif norm == "z":       # each neuron's rate z-scored over positions
            v = (v - v.mean()) / (v.std() if v.std() > 0 else 1)
        rows.append(v)
    X = np.concatenate(rows, axis=0)
    # X: (neurons x conditions) x positions -> correlation between positions across the vector entries
    X = np.nan_to_num(X)
    C = np.corrcoef(X.T)
    ws = []
    for i in range(C.shape[0]):
        a = b = i
        while a > 0 and C[a - 1, i] >= 0.5:
            a -= 1
        while b < C.shape[0] - 1 and C[b + 1, i] >= 0.5:
            b += 1
        ws.append((b - a + 1) * dx)
    return np.mean(ws), np.std(ws, ddof=1)
PVW = []
for pr, nm in itertools.product(("P-b", "P-a"), ("raw", "peak", "z")):
    for rd in "ab":
        m2, s2 = pv_width([k for k in KEYS if k[0] == "EE188"], rd, pr, nm)
        m5, s5 = pv_width([k for k in KEYS if k[0] == "FF130"], rd, pr, nm)
        PVW.append((pr, nm, rd, m2, s2, m5, s5))
        print(f"{pr} {nm} ({rd}): EE188 {m2:.3f} +/- {s2:.3f} (0.18 +/- 0.087: {abs(m2-0.18)<=0.087}; s.d. against 0.087 and 0.126); "
              f"FF130 stand-in {m5:.3f} +/- {s5:.3f} (0.27 +/- 0.13: {abs(m5-0.27)<=0.13})")
        for name, l5 in LAYER_READINGS.items():
            a = pv_width(keys_layer("L2/3", l5), rd, pr, nm); b = pv_width(keys_layer("L5", l5), rd, pr, nm)
            print(f"    all sessions {name}: layer 2/3 {a[0]:.3f} +/- {a[1]:.3f} ({abs(a[0]-0.18)<=0.087}), layer 5 {b[0]:.3f} +/- {b[1]:.3f} ({abs(b[0]-0.27)<=0.13})")

**Result: partially reproduced (resting on an assumption).** With raw rates the correlation stays above half everywhere (width about 1.0), so the measure needs a normalisation of each neuron's rate that the paper does not state. With each neuron's rate normalised to its peak (the assumption, as Figure 2a displays the rates), layer 5 under R-a is within tolerance (FF.130 0.358 to 0.398, all FF sessions 0.324 to 0.359), and layer 2/3 is just outside (EE.188 0.268 to 0.314, against the limit of 0.267). With rates z-scored per neuron both layers are within tolerance (EE.188 0.185 to 0.219; FF 0.208 to 0.238). Under the adopted normalisation layer 5 matches and layer 2/3 does not under any reading: the measured value (P-b, reading (a)) is 0.282, 0.015 above the limit, and the closest (reading (b)) is 0.268, 0.001 above it. One of the two values matches, so the outcome is partial.

## Claim fig2d-layer-rates

> Layer 5 neurons fire at higher mean rates than layer 2/3 neurons, over all neurons (222 against 411) and over putative pyramidal cells (85 against 133), both at P < 0.01 by t-test; putative interneurons (47 against 37) carry no significance mark.

Mean rate per neuron over (a) trials only, (b) the whole task session (first to last spike of the session's units). Cell types from B-a, W-a (C-a). Student's and Welch's t-tests, under each rat reading.

In [ ]:
def mean_rate(k, period):
    s, u = k
    x = S[s]; t = x.times[u]
    if period == "a":
        return np.sum(x.trial_of(t) >= 0) / np.sum(x.stop - x.start)
    T0_, T1_ = min(v[0] for v in x.times if len(v)), max(v[-1] for v in x.times if len(v))
    return len(t) / (T1_ - T0_)
RATE = {(k, p): mean_rate(k, p) for k in KEYS for p in "ab"}
for name, l5 in LAYER_READINGS.items():
    for p in "ab":
        out = []
        for grp in ("all", "pyr", "int"):
            g5 = [RATE[(k, p)] for k in keys_layer("L5", l5) if grp == "all" or TY[k] == grp]
            g2 = [RATE[(k, p)] for k in keys_layer("L2/3", l5) if grp == "all" or TY[k] == grp]
            ts = st.ttest_ind(g5, g2) if len(g5) > 1 and len(g2) > 1 else None
            tw = st.ttest_ind(g5, g2, equal_var=False) if len(g5) > 1 and len(g2) > 1 else None
            out.append((grp, np.mean(g5) if g5 else np.nan, np.std(g5) if g5 else np.nan, len(g5), np.mean(g2), np.std(g2), len(g2),
                        ts.pvalue if ts else np.nan, tw.pvalue if tw else np.nan))
        print(f"{name}, period ({p}): " + "; ".join(f"{g}: L5 {m5:.2f}+/-{s5:.2f} Hz (n={n5}) vs L2/3 {m2:.2f}+/-{s2:.2f} (n={n2}), Student p={ps:.2g}, Welch p={pw:.2g}"
                                               for g, m5, s5, n5, m2, s2, n2, ps, pw in out))
        ok = [o[1] > o[4] and min(o[7], o[8]) < 0.01 for o in out[:2]]
        print("    layer 5 higher at p < 0.01 (all, pyramidal):", ok)
# variant on the unit list: every NWB mPFC unit (all neurons only, as the cell types are found for the listed units)
print("variant, every NWB mPFC unit, all neurons:")
for name, l5 in LAYER_READINGS.items():
    for p in "ab":
        g = {"L5": [], "L2/3": []}
        for s, x in S.items():
            T0_, T1_ = min(v[0] for v in x.times if len(v)), max(v[-1] for v in x.times if len(v))
            for t in NWBU[s]["times"]:
                r_ = np.sum(x.trial_of(t) >= 0) / np.sum(x.stop - x.start) if p == "a" else len(t) / (T1_ - T0_)
                g[layer_of(s[:2], l5)].append(r_)
        ts, tw = st.ttest_ind(g["L5"], g["L2/3"]), st.ttest_ind(g["L5"], g["L2/3"], equal_var=False)
        print(f"   {name}, period ({p}): L5 {np.mean(g['L5']):.2f} Hz (n={len(g['L5'])}) vs L2/3 {np.mean(g['L2/3']):.2f} (n={len(g['L2/3'])}), Student p={ts.pvalue:.2g}, Welch p={tw.pvalue:.2g}")

**Result: partially reproduced.** Under R-a, trial rates, layer 5 pyramidal cells fire faster than layer 2/3 ones (3.02 against 1.74 Hz, Student p = 0.0028), but over all neurons the difference (3.05 against 2.28 Hz) reaches p = 0.026 (Welch), not 0.01. Over the whole session neither reaches 0.01. Interneurons do not differ (5.29 against 5.30 Hz). With every NWB mPFC unit, all neurons differ strongly under R-a (2.77 against 0.99 Hz, Student p = 7.6e-13), but that list has no cell types, so the pyramidal comparison cannot be made on it; under it too one of the two parts is shown to hold.

## Claim fig2e-layer-discrimination

> Single layer 5 neurons discriminate left from right trials over a larger fraction of the maze than layer 2/3 neurons, for all neurons (P < 0.05), putative pyramidal cells (P < 0.01) and putative interneurons (P < 0.01), by t-test.

Per neuron, the total length of its significant segments as a fraction of the maze; readings (a) all neurons (0 for a non-selective one), (b) selective neurons only. Types from B-a, W-a (C-a).

In [ ]:
LEN = {k: fj.seg_length(SEL[k]["segments"]) for k in KEYS}
for name, l5 in LAYER_READINGS.items():
    for rd in "ab":
        out = []
        for grp, lvl in (("all", 0.05), ("pyr", 0.01), ("int", 0.01)):
            sel = lambda k: (grp == "all" or TY[k] == grp) and (rd == "a" or SEL[k]["significant"])
            g5 = [LEN[k] for k in keys_layer("L5", l5) if sel(k)]
            g2 = [LEN[k] for k in keys_layer("L2/3", l5) if sel(k)]
            if len(g5) > 1 and len(g2) > 1:
                ps, pw = st.ttest_ind(g5, g2).pvalue, st.ttest_ind(g5, g2, equal_var=False).pvalue
            else:
                ps = pw = np.nan
            out.append((grp, lvl, np.mean(g5) if g5 else np.nan, len(g5), np.mean(g2) if g2 else np.nan, len(g2), ps, pw))
        print(f"{name} ({rd}): " + "; ".join(f"{g}: L5 {m5:.3f} (n={n5}) vs L2/3 {m2:.3f} (n={n2}), Student p={ps:.2g}, Welch p={pw:.2g}"
                                         for g, l, m5, n5, m2, n2, ps, pw in out))
        print("    higher at the paper's level:", [o[2] > o[4] and min(o[6], o[7]) < o[1] for o in out],
              "; higher at p < 0.05:", [o[2] > o[4] and min(o[6], o[7]) < 0.05 for o in out])

**Result: not reproduced.** Under R-a only pyramidal cells discriminate over a larger fraction in layer 5 at p < 0.05 (0.331 against 0.244, Student p = 0.023, reading (a)); all neurons (p = 0.077) and interneurons (4 against 19 cells, layer 5 lower) do not. Under R-b no group does.

## Claim traj-diverge

> Left and right running paths overlap in the central arm and diverge significantly from position 0.30; head directions diverge from 0.24.

Per trial, the head position (midpoint of the two LEDs, pixels; the image scale is not documented, which does not change where a difference starts) projected on the axis perpendicular to the central arm (the mean direction from position 0 to 0.3 over the session's trials), and the head direction (angle of the line from LED 1 to LED 2, relative to the central arm, wrapped), each interpolated at the grid of normalised positions from the tracked frames of the trial. Left minus right mean, the permutation test at global level 0.01, with M = 10,000 permutations: the paper gives no M, and with 1,000 the discrete stepping procedure at level 0.01 settles on the outermost band (the data cannot break it), as shown in the next cell's output for EE.188. EE.188 (Supplementary Figure 3A uses the Figure 2a layer 2/3 session), every session as a variant. Checks of the head direction follow: the LED 1 to LED 2 axis against the direction of movement, the mean angles of left and right trials along the maze, and the same test with each angle taken relative to the circular mean of all trials at that position, which cannot wrap.

In [ ]:
def head_curves(x):
    l1 = x.leds[0]
    two = len(x.leds) > 1
    l2 = x.leds[1] if two else np.full_like(l1, np.nan)
    mid = (l1 + l2) / 2 if two else l1
    lat, hd, keep = [], [], []
    # central arm direction from position 0 to 0.3
    p0, p3 = [], []
    for j, (a, b) in enumerate(zip(x.start, x.stop)):
        fr = np.nonzero((x.whl_t >= a) & (x.whl_t <= b) & (x.d > 0) & np.isfinite(mid[:, 0]))[0]
        if len(fr) < 5:
            continue
        pos = x.d[fr] - 1
        o = np.argsort(pos)
        p0.append(mid[fr[o[0]]]); 
        k3 = fr[np.argmin(np.abs(pos - 0.3))]; p3.append(mid[k3])
    u = np.mean(p3, 0) - np.mean(p0, 0); u /= np.linalg.norm(u)
    perp = np.array([-u[1], u[0]])
    ang0 = np.arctan2(u[1], u[0])
    for j, (a, b) in enumerate(zip(x.start, x.stop)):
        fr = np.nonzero((x.whl_t >= a) & (x.whl_t <= b) & (x.d > 0) & np.isfinite(mid[:, 0]) & (np.isfinite(l2[:, 0]) | (not two)))[0]
        if len(fr) < 5:
            continue
        pos = x.d[fr] - 1
        o = np.argsort(pos)
        lv = mid[fr] @ perp
        v = l2[fr] - l1[fr]
        an = np.angle(np.exp(1j * (np.arctan2(v[:, 1], v[:, 0]) - ang0)))
        lat.append(np.interp(fj.GRID, pos[o], lv[o]))
        # unwrap along position before interpolating
        an_o = np.unwrap(an[o])
        hd.append(np.angle(np.exp(1j * np.interp(fj.GRID, pos[o], an_o))))
        keep.append(j)
    return np.array(lat), np.array(hd), np.array(keep), two
TRAJ, HD_GLOB = {}, {}
for si, (s, x) in enumerate(S.items()):
    lat, hd, keep, two = head_curves(x)
    lab = x.side[keep] == "L"
    rng = np.random.default_rng(500 + si)
    r1 = fj.perm_bands(lat, lab, M=10000, alpha=0.01, rng=rng)
    f1 = fj.GRID[np.argmax(r1["glob"])] if r1["glob"].any() else np.nan
    if two:
        r2 = fj.perm_bands(hd, lab, M=10000, alpha=0.01, rng=rng)
        f2 = fj.GRID[np.argmax(r2["glob"])] if r2["glob"].any() else np.nan
        HD_GLOB[s] = r2["glob"]
    else:
        f2 = np.nan     # one LED only: no head direction
    TRAJ[s] = (f1, f2, len(keep))
    print(f"{s}: LEDs {len(x.leds)}; trials with tracking {len(keep)}/{x.n_trials}; first globally significant position: path {f1:.2f}, head direction {f2:.2f}; "
          f"|L-R| path difference at 0, 0.3, 0.6: {np.abs(r1['D0'][[0, 30, 60]]).round(1).tolist()} px")
lat, hd, keep, two = head_curves(S["EE188"])
r = fj.perm_bands(lat, S["EE188"].side[keep] == "L", M=1000, alpha=0.01, rng=np.random.default_rng(1))
print("EE188 path with M = 1,000: global band index", r["kstar"], "; globally significant positions", int(r["glob"].sum()))
f1, f2, _ = TRAJ["EE188"]
print(f"EE188: path {f1:.2f} (0.30 +/- 0.03: {abs(f1-0.30)<=0.03}); head direction {f2:.2f} (0.24 +/- 0.024: {abs(f2-0.24)<=0.024})")
# Checks of the head direction (verifier, round 1): LED order and axis, the angles behind the early difference, and a
# reading free of wrapping (each angle taken relative to the circular mean of all trials at that position)
def led_axis_check(x):
    """Angle of the LED 1 -> LED 2 axis relative to the direction of movement in the central arm (positions 0.05 to 0.3)."""
    l1, l2 = x.leds[0], x.leds[1]
    mid = (l1 + l2) / 2
    pos = x.d - 1
    ok = (x.trial_of(x.whl_t) >= 0) & (x.d > 0) & (pos > 0.05) & (pos < 0.3) & np.isfinite(l1[:, 0]) & np.isfinite(l2[:, 0])
    i = np.nonzero(ok)[0]
    i = i[i + 4 < len(mid)]
    v = mid[i + 4] - mid[i]                    # displacement over 0.1 s
    keep = np.linalg.norm(v, axis=1) > 5       # moving frames (> 5 px per 0.1 s) [standard]
    ax = l2[i] - l1[i]
    da = np.exp(1j * (np.arctan2(ax[keep, 1], ax[keep, 0]) - np.arctan2(v[keep, 1], v[keep, 0])))
    return np.degrees(np.angle(da.mean())), np.abs(da.mean()), int(keep.sum()), float(np.median(np.linalg.norm(ax, axis=1)))
cmean = lambda a: np.degrees(np.angle(np.mean(np.exp(1j * a), axis=0)))
gi = [0, 2, 5, 10, 15, 20, 24, 30, 40]
for si, s in enumerate(("EE152", "EE165", "EE188")):
    x = S[s]
    m_, R_, n_, sep = led_axis_check(x)
    lat, hd, keep, two = head_curves(x)
    lab = x.side[keep] == "L"
    print(f"{s}: LED 1 -> LED 2 axis against the direction of movement, circular mean {m_:.0f} deg, resultant length {R_:.2f} ({n_} frames); LED separation {sep:.1f} px")
    print(f"   head angle relative to the central arm at positions {[float(fj.GRID[i]) for i in gi]}: left {np.round(cmean(hd[lab])[gi]).tolist()} deg, "
          f"right {np.round(cmean(hd[~lab])[gi]).tolist()} deg; share of angles beyond +/-150 deg at 0 to 0.3: {np.mean(np.abs(hd[:, :31]) > np.radians(150)):.3f}")
    mu = np.angle(np.mean(np.exp(1j * hd), axis=0))
    hc = np.angle(np.exp(1j * (hd - mu)))
    r3 = fj.perm_bands(hc, lab, M=10000, alpha=0.01, rng=np.random.default_rng(600 + si))
    g_ = r3["glob"]
    starts = [round(float(fj.GRID[i]), 2) for i in np.nonzero(np.diff(np.r_[0, g_.astype(int)]) == 1)[0]]
    g0 = HD_GLOB[s]
    starts0 = [round(float(fj.GRID[i]), 2) for i in np.nonzero(np.diff(np.r_[0, g0.astype(int)]) == 1)[0]]
    print(f"   globally significant runs of head direction start at {starts0} (angles as above) and at {starts} (re-centred on the "
          f"circular mean of all trials at each position); first position {starts0[0] if starts0 else np.nan} and {starts[0] if starts else np.nan}")

**Result: partially reproduced.** In EE.188 the lateral head position first differs significantly at 0.30 (expected 0.30 +/- 0.03), while head direction already differs at 0.02, not 0.24. The early difference is not an artefact of the angle: the LED 1 to LED 2 axis points along the direction of movement (circular mean -5 deg, resultant length 0.98 in EE.188), the angles near the start are far from the wrap at +/-180 deg (left -95 and -82 deg, right -121 and -121 deg at 0 and 0.02), and the re-centred angles, which cannot wrap, give 0.03. It is the turn out of the start box, where the head points about 80 to 120 deg away from the central arm and differs by side; the two headings then meet near 0.1 (-10 and -11 deg) and a second significant run begins at 0.23 under both angle readings, within 0.24 +/- 0.024. The criterion takes the first significant position, so head direction does not match. Head direction exists only in the three sessions with two LEDs (EE.152, EE.165, EE.188).

## Claim odor-nosepoke

> About a quarter of neurons (26% in one layer 2/3 session) fire at significantly different rates during nose poking for the two odors.

Per neuron, its rate during each trial's nose poking, left against right trials, t-test at p < 0.05. Nose poking: (q) from the start of the first poke to the end of the last (SessionNP columns 1 and 2); (p) the time the trial's sensor (channel 97 right, 98 left) is active between those two times, summed (sensor envelope above 20% of its median during-poke level; only where the sensor channels were found, so not GG nor EE.088). Neurons (a) EE.188 (the Figure 2a session, as Supplementary Figure 6B), (b) all neurons. Student's and Welch's tests; a neuron without spikes in any poke has no test and counts as not significant.

In [ ]:
def poke_rates(x, reading):
    if reading == "q":
        iv = [[(a, b)] for a, b in zip(x.poke_start, x.start)]
    else:
        if LINK.get(x.name) is None or x.rat == "GG":
            return None
        E = LINK[x.name][2]
        iv = []
        for a, b, sd in zip(x.poke_start, x.start, x.side):
            e = E[int(a * 1000):int(b * 1000), 0 if sd == "R" else 1]
            on = e > 0.2 * np.median(e)
            d = np.diff(np.r_[0, on.astype(int), 0])
            st_, en_ = np.nonzero(d == 1)[0], np.nonzero(d == -1)[0]
            iv.append([(a + p / 1000, a + q / 1000) for p, q in zip(st_, en_)] or [(a, b)])
    dur = np.array([sum(b - a for a, b in v) for v in iv])
    R = np.array([[sum(np.sum((t >= a) & (t < b)) for a, b in v) for v in iv] for t in x.times]) / dur
    return R, dur
ODOR = {}
for rd in "qp":
    for s, x in S.items():
        r = poke_rates(x, rd)
        if r is None:
            continue
        R, dur = r
        lab = x.side == "L"
        for u in range(len(x.cells)):
            a_, b_ = R[u, lab], R[u, ~lab]
            if R[u].sum() == 0:
                ODOR[(rd, s, u)] = (np.nan, np.nan, a_.mean() - b_.mean())
                continue
            ODOR[(rd, s, u)] = (st.ttest_ind(a_, b_).pvalue, st.ttest_ind(a_, b_, equal_var=False).pvalue, a_.mean() - b_.mean())
for rd in "qp":
    for nm, pick in (("(a) EE188", lambda s: s == "EE188"), ("(b) all", lambda s: True)):
        ks = [k for k in ODOR if k[0] == rd and pick(k[1])]
        for ti, tn in ((0, "Student"), (1, "Welch")):
            sig = sum(np.nan_to_num(ODOR[k][ti], nan=1) < 0.05 for k in ks)
            f = pct(sig, len(ks))
            print(f"({rd}) {nm} {tn}: {sig}/{len(ks)} = {f:.1f}% (26 +/- 2.6: {abs(f-26)<=2.6}); no test (no spikes): {sum(np.isnan(ODOR[k][0]) for k in ks)}")
x = S["EE188"]
print("EE188 poke duration (q) median: left %.2f s, right %.2f s" % (np.median((x.start - x.poke_start)[x.side == 'L']), np.median((x.start - x.poke_start)[x.side == 'R'])))
r = poke_rates(x, "p"); print("EE188 poke duration (p) median: left %.2f s, right %.2f s" % (np.median(r[1][x.side == 'L']), np.median(r[1][x.side == 'R'])))

**Result: not reproduced.** In EE.188, 45.3% (Student) and 42.7% (Welch) of neurons fire differently during left and right nose pokes from first poke start to last poke end; over all neurons 32.4% and 32.9%; with the sensor-timed pokes (EE and FF sessions) the fractions are between 31% and 44%. None is within 26 +/- 2.6.

## Claim odor-unrelated-to-stem

> Neurons' odor preference during nose poking is unrelated to their left-right differences in the stem and side arms, so odor traces do not explain the stem selectivity.

In EE.188 (the session of odor-nosepoke reading (a)), with odor-selective neurons from reading (q), Student's t-test: (a) Fisher exact test of odor-selective against stem- (0 to 0.3) or side-arm- (0.5 to 1) selective; (b) Pearson correlation across neurons between the nose-poke rate difference (left minus right) and the mean of D0 over 0 to 0.3 or 0.5 to 1. Reproduced when no test finds an association at p < 0.05. Other readings of odor selectivity as variants.

In [ ]:
def odor_vs_maze(rd, ti, sessions):
    ks = [k for k in KEYS if k[0] in sessions]
    od = np.array([np.nan_to_num(ODOR[(rd, s, u)][ti], nan=1) < 0.05 for s, u in ks])
    dif = np.array([ODOR[(rd, s, u)][2] for s, u in ks])
    out = []
    for nm, lo, hi in (("stem 0-0.3", 0, 0.3), ("side 0.5-1", 0.5, 1)):
        ms = np.array([fj.seg_in(SEL[k]["segments"], lo, hi) for k in ks])
        tab = [[np.sum(od & ms), np.sum(od & ~ms)], [np.sum(~od & ms), np.sum(~od & ~ms)]]
        orr, pf = st.fisher_exact(tab)
        g = (fj.GRID >= lo) & (fj.GRID <= hi)
        md = np.array([SEL[k]["D0"][g].mean() for k in ks])
        r, pr = st.pearsonr(dif, md)
        out.append((nm, tab, orr, pf, r, pr))
    return out
for rd, ti, sess, lab_ in (("q", 0, ("EE188",), "main: EE188, (q), Student"), ("q", 1, ("EE188",), "EE188, (q), Welch"),
                           ("p", 0, ("EE188",), "EE188, (p), Student"), ("q", 0, tuple(S), "all sessions, (q), Student")):
    for nm, tab, orr, pf, r, pr in odor_vs_maze(rd, ti, sess):
        print(f"{lab_} | {nm}: odor x maze table {tab}, odds ratio {orr:.2f}, Fisher p {pf:.3f}; correlation r = {r:.2f}, p = {pr:.3g}")

**Result: not reproduced.** In EE.188 the Fisher tests find no association of odor selectivity with stem (p = 0.705) or side-arm selectivity (p = 0.576), but the nose-poke rate difference correlates with the mean left-right difference in the stem (r = -0.38, p = 2e-5) and in the side arms (r = -0.20, p = 0.032).

## Claim retrospective-coding-rare

> Only a small fraction of neurons fire differently according to the reward location of the previous trial.

The left-right test with labels from the previous listed trial (the first trial left out). SessionNP lists correct trials only, so the previous listed trial is the previous trial only when no error or aborted trial came between: a deviation the data impose. Reproduced when, in each layer, the fraction for previous-trial labels is less than half the fraction for current labels. Readings (a) the Figure 2a session (EE.188; FF.130 stand-in for layer 5), (b) all sessions of each layer under each rat reading.

In [ ]:
def frac_sig(keys, res):
    h = sum(res[k]["significant"] for k in keys)
    return pct(h, len(keys)), h, len(keys)
for nm, ks in (("EE188", [k for k in KEYS if k[0] == "EE188"]), ("FF130 stand-in", [k for k in KEYS if k[0] == "FF130"])):
    a, b = frac_sig(ks, PREV), frac_sig(ks, SEL)
    print(f"(a) {nm}: previous-trial labels {a[0]:.1f}% ({a[1]}/{a[2]}), current {b[0]:.1f}% ({b[1]}/{b[2]}); less than half: {a[0] < b[0] / 2}")
for name, l5 in LAYER_READINGS.items():
    out = []
    for layer in ("L2/3", "L5"):
        ks = keys_layer(layer, l5)
        a, b = frac_sig(ks, PREV), frac_sig(ks, SEL)
        out.append(f"{layer}: previous {a[0]:.1f}% ({a[1]}/{a[2]}), current {b[0]:.1f}%, less than half {a[0] < b[0] / 2}")
    print(f"(b) {name}: " + "; ".join(out))
print("null relabelling, all neurons: %.1f%%" % frac_sig(KEYS, NULL1)[0])

**Result: reproduced.** With previous-trial labels 4.3% of EE.188 neurons are significant against 60.7% with current labels, and under every rat reading each layer's previous-trial fraction (5.7% to 8.9%) is below half its current fraction (51.4% to 66.4%); the null relabelling gives 4.8%.

## Claim speed-not-correlated

> Firing rates show no net positive relationship with running speed; the distribution of rate-speed correlations is symmetric around zero.

Supplementary Figure 3C: trials divided into 0.5 s bins; per bin the neuron's rate and the running speed (path length of the head position, the LED midpoint where two LEDs exist, in pixels per second); Pearson correlation per neuron over its session's bins. Two-sided Wilcoxon signed-rank test of the correlations against 0, over all neurons and within putative pyramidal cells and interneurons (B-a, W-a). Reproduced when the median over all neurons is not significantly above 0 at p < 0.05.

In [ ]:
SPD = {}
for s, x in S.items():
    l1 = x.leds[0]
    mid = (l1 + x.leds[1]) / 2 if len(x.leds) > 1 else l1
    edges = np.concatenate([np.arange(a, b - 0.5 + 1e-9, 0.5) for a, b in zip(x.start, x.stop)])
    sp = []
    for e0 in edges:
        fr = np.nonzero((x.whl_t >= e0) & (x.whl_t < e0 + 0.5))[0]
        p = mid[fr]
        p = p[np.isfinite(p[:, 0])]
        sp.append(np.sum(np.linalg.norm(np.diff(p, axis=0), axis=1)) / 0.5 if len(p) > 2 else np.nan)
    sp = np.array(sp)
    ok = np.isfinite(sp)
    for u, t in enumerate(x.times):
        idx = np.searchsorted(t, edges); idx2 = np.searchsorted(t, edges + 0.5)
        r = (idx2 - idx) / 0.5
        SPD[(s, u)] = st.pearsonr(r[ok], sp[ok])[0] if r[ok].std() > 0 else np.nan
for grp in ("all", "pyr", "int"):
    v = np.array([SPD[k] for k in KEYS if (grp == "all" or TY[k] == grp) and np.isfinite(SPD[k])])
    w = st.wilcoxon(v)
    print(f"{grp}: n = {len(v)}, median r = {np.median(v):.3f}, mean {np.mean(v):.3f}, range [{v.min():.2f}, {v.max():.2f}], "
          f"positive {np.mean(v > 0)*100:.0f}%, Wilcoxon p = {w.pvalue:.3g}")
print("neurons without a correlation (no spikes in trial bins):", sum(not np.isfinite(SPD[k]) for k in KEYS))
v = np.array([SPD[k] for k in KEYS if np.isfinite(SPD[k])])
print("median significantly above 0:", bool(np.median(v) > 0 and st.wilcoxon(v).pvalue < 0.05))

**Result: reproduced.** The median rate-speed correlation over 627 neurons is -0.022 (44% positive), not above 0; the Wilcoxon test finds it below 0 (p = 7.5e-05), and within pyramidal cells (median -0.022, p = 0.016) and interneurons (-0.046, p = 0.25).

## Position of excess coincidences (shared by claims fig4-position-dependence, fig5a, fig4-pair-156-136, fig5b-example-pairs)

Methods, "Identifying position dependence of monosynaptic interactions": coincident events are spike pairs with 1 <= t_post - t_pre <= 4 ms in trials, placed at the presynaptic spike's position, smoothed with the 0.05 kernel and divided by the number of trials; surrogates by jittering both cells' spikes on [-5, +5] ms; one-sided pointwise and global bands for excess at P < 0.01. M = 5,000 surrogates: the section gives no number, and with 1,000 the discrete global band at level 0.01 cannot be broken in about half of the tests (shown below). Each excitatory connection (B-a, W-a) is tested on left trials, right trials and all trials.

In [ ]:
POS = {}
t1 = time.time()
EXC = [(s, a, b) for s, a, b, k in CONN[MAIN] if k == "E"]
for q, (s, a, b) in enumerate(EXC):
    x = S[s]
    for cond, m in (("L", x.side == "L"), ("R", x.side == "R"), ("all", np.ones(x.n_trials, bool))):
        POS[(s, a, b, cond)] = fj.coincidence_position_test(x, a, b, m, M=5000, alpha=0.01, seed=20000 + q)
print(f"{len(EXC)} excitatory connections tested in {time.time() - t1:.0f} s; tests with no coincidences in trials:",
      sum(v is None for v in POS.values()), "; peak memory so far %.1f GB" % (resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6))
x = S["EE188"]; c = list(x.cells)
r1000 = [fj.coincidence_position_test(S[s], a, b, np.ones(S[s].n_trials, bool), M=1000, alpha=0.01, seed=1) for s, a, b in EXC[::20]]
print("with M = 1,000 (every 20th connection, all trials): global band at the outermost surrogate (cannot be broken) in",
      sum(r is not None and r["kstar"] == 0 for r in r1000), "of", sum(r is not None for r in r1000))
def posdep(r, rd):
    if r is None or not r["significant"]:
        return False
    if rd == "i":
        return True
    return bool(np.any(r["sign"] == 0))

## Claim fig4-position-dependence

> Of 343 pairs with significant excitatory monosynaptic connections, 67 show identifiable position dependence of excess short-latency coincidences in the maze.

Fraction 67/343 = 19.5% +/- 1.95. Trials (a) left and right apart, a pair counting when either shows dependence; (b) all trials. Dependence (i) a globally significant region; (ii) and a part of the maze where the pointwise band is not broken.

In [ ]:
FIG4 = {}
for tr in "ab":
    for rd in ("i", "ii"):
        hits = [(posdep(POS[(s, a, b, "L")], rd) or posdep(POS[(s, a, b, "R")], rd)) if tr == "a" else posdep(POS[(s, a, b, "all")], rd)
                for s, a, b in EXC]
        k, n = int(np.sum(hits)), len(hits)
        null = 0.02 if tr == "a" else 0.01
        pb = st.binomtest(k, n, null, alternative="greater").pvalue
        FIG4[(tr, rd)] = (k, n, pct(k, n))
        print(f"trials ({tr}), dependence ({rd}): {k}/{n} = {pct(k, n):.1f}% (19.5 +/- 1.95: {abs(pct(k, n)-19.5)<=1.95}); "
              f"against the null rate {null:.0%}: one-sided binomial p = {pb:.3g}")
print("per session, (b)(i):", {s: f"{sum(posdep(POS[(ss, a, b, 'all')], 'i') for ss, a, b in EXC if ss == s)}/{sum(1 for ss, a, b in EXC if ss == s)}" for s in S})

**Result: partially reproduced (resting on an assumption).** Of 472 excitatory connections, 30 (6.4%, left and right apart) or 34 (7.2%, all trials) show position-dependent excess coincidences, below 19.5 +/- 1.95% but above the null rates of 2% and 1% (binomial p = 5e-08 and 1e-18). The test uses 5,000 jitter surrogates: with 1,000 the global band cannot be broken in 17 of 23 connections checked.

## Claim fig5a-session-selective-connections

> In the layer 2/3 session of Figure 3d, 15 of 36 excitatory connections are active selectively in maze segments of left or right trajectories.

EE.188, the readings of fig4-position-dependence. Sets of connections: (1) the excitatory connections found here; (2) the lab's list pairs_all in EE188_example.mat (the connections of the paper's Figure 3d session), tested with the same position test where they are not already among (1).

In [ ]:
x = S["EE188"]; c = list(x.cells)
lab_list = [(CIDX[a], CIDX[b]) for a, b in ex["pairs_all"].astype(int)]
lab_exc = [(a, b) for a, b in lab_list if S["EE188"].lab_type[a] == 1]
for (a, b) in lab_list:
    for cond, m in (("L", x.side == "L"), ("R", x.side == "R"), ("all", np.ones(x.n_trials, bool))):
        if ("EE188", a, b, cond) not in POS:
            POS[("EE188", a, b, cond)] = fj.coincidence_position_test(x, a, b, m, M=5000, alpha=0.01, seed=31000 + a * 300 + b)
for nm, pairs in (("(1) excitatory connections found here", [(a, b) for s, a, b in EXC if s == "EE188"]),
                  ("(2) lab pairs_all", lab_list), ("(2') lab pairs_all with a lab-pyramidal reference", lab_exc)):
    for tr in "ab":
        for rd in ("i", "ii"):
            k = sum((posdep(POS[("EE188", a, b, "L")], rd) or posdep(POS[("EE188", a, b, "R")], rd)) if tr == "a"
                    else posdep(POS[("EE188", a, b, "all")], rd) for a, b in pairs)
            n = len(pairs)
            print(f"{nm}, trials ({tr}), ({rd}): {k} of {n} (expected 15 +/- 1.5 of 36 +/- 3.6: {abs(k-15)<=1.5 and abs(n-36)<=3.6}; "
                  f"fraction {pct(k, n):.1f}% vs 41.7 +/- 4.2: {abs(pct(k, n)-41.7)<=4.2})")

**Result: not reproduced (resting on an assumption).** In EE.188, 8 or 9 of the 119 excitatory connections found here are position dependent; of the lab's 43 listed connections, 6 or 7 (14.0% to 17.5% of the set), against 15 of 36 (41.7%).

## Claim fig4-pair-156-136

> Pair 156-136, on different shanks, keeps steady firing rates between positions 0.2 and 0.5 on left trials, yet excess short-latency coincidences occur only between 0.4 and 0.6.

Cells 156 and 136 of EE.188 (the CRCNS cell numbers, which are the paper's: the example file's cells carry the numbers of Figures 3 to 7), left trials. Steady: each cell's left-trial PSTH between 0.2 and 0.5 varies by less than 50% of its maximum there.

In [ ]:
def pair_report(pre, post, cond):
    a, b = c.index(pre), c.index(post)
    r = POS.get(("EE188", a, b, cond)) or fj.coincidence_position_test(x, a, b, x.side == cond if cond != "all" else np.ones(x.n_trials, bool), M=5000, alpha=0.01, seed=pre * 1000 + post)
    POS[("EE188", a, b, cond)] = r
    conn = [k for s, aa, bb, k in CONN[MAIN] if s == "EE188" and aa == a and bb == b]
    return a, b, r, conn
a, b, r, conn = pair_report(156, 136, "L")
print("shanks:", x.shank[a], x.shank[b], "; lab types", x.lab_type[a], x.lab_type[b], "; connection found here:", conn,
      "; in the lab list:", (a, b) in lab_list)
g = (fj.GRID >= 0.2) & (fj.GRID <= 0.5)
for u, nm in ((a, "156"), (b, "136")):
    v = PSTH[("EE188", u)][x.side == "L"].mean(0)[g]
    print(f"cell {nm}: left PSTH 0.2-0.5 from {v.min():.2f} to {v.max():.2f} (variation {(v.max()-v.min())/v.max()*100:.0f}% of max; steady: {(v.max()-v.min()) < 0.5*v.max()})")
print("significant regions (left trials):", [(round(p, 2), round(q, 2)) for p, q, s_ in r["segments"]], "; coincident events", r["n_events"])
if r["segments"]:
    p, q, _ = r["segments"][0]
    print(f"only one region: {len(r['segments']) == 1}; start {p:.2f} (0.4 +/- 0.04: {abs(p-0.4)<=0.04+1e-9}), end {q:.2f} (0.6 +/- 0.06: {abs(q-0.6)<=0.06+1e-9})")
for jb in (False,):
    rr = fj.coincidence_position_test(x, a, b, x.side == "L", M=5000, alpha=0.01, seed=5, jitter_b=jb)
    print("variant, jitter of the presynaptic cell only (Supplementary Fig. 12B):", [(round(p, 2), round(q, 2)) for p, q, s_ in rr["segments"]])

**Result: not reproduced (resting on an assumption).** Cells 156 and 136 are on shanks 5 and 4 and connected (excitatory, and in the lab's list). On left trials the only significant region runs from 0.34 to 0.64: the end is within 0.6 +/- 0.06, the start is outside 0.4 +/- 0.04. Cell 156's rate between 0.2 and 0.5 varies by 49% of its maximum (steady), cell 136's by 61% (not steady).

## Claim fig5b-example-pairs

> In pair 197-201 transmission is significant only between 0.4 and 0.5 of left trials although the expected coincidence rate is as high from 0 to 0.5; pair 49-52 transmits only on left trials; pair 186-201 transmits only toward the end of the right arm although expected coincidences are higher earlier.

In [ ]:
res5 = {}
for pre, post in ((197, 201), (49, 52), (186, 201)):
    for cond in ("L", "R"):
        a, b, r, conn = pair_report(pre, post, cond)
        res5[(pre, post, cond)] = r
        segs = [(round(p, 2), round(q, 2)) for p, q, s_ in r["segments"]] if r else None
        exp = r["expected"] if r else None
        print(f"{pre}-{post} {cond}: connection here {conn}, lab list {(a, b) in lab_list}; regions {segs}; "
              f"expected coincidence rate at 0.1/0.3/0.45/0.8: {np.round(exp[[10, 30, 45, 80]], 3).tolist() if exp is not None else None}")
r = res5[(197, 201, "L")]
ok1 = bool(r and len(r["segments"]) == 1 and abs(r["segments"][0][0] - 0.4) <= 0.04 + 1e-9 and abs(r["segments"][0][1] - 0.5) <= 0.05 + 1e-9)
ok2 = bool(res5[(49, 52, "L")] and res5[(49, 52, "L")]["significant"] and not (res5[(49, 52, "R")] and res5[(49, 52, "R")]["significant"]))
rR, rL = res5[(186, 201, "R")], res5[(186, 201, "L")]
ok3 = bool(rR and len(rR["segments"]) == 1 and rR["segments"][0][0] > 0.75 and not (rL and rL["significant"]))
print("197-201 matches:", ok1, "; 49-52 left only:", ok2, "; 186-201 only on right beyond 0.75:", ok3)

**Result: not reproduced (resting on an assumption).** Pair 197-201 has its only significant left-trial region from 0.36 to 0.51, within tolerance. Pair 49-52 has no significant region on left or right trials. Pair 186-201 has its only region on right trials from 0.73 to 1.0 and none on left trials, starting before the 0.75 the criterion sets. One of three pairs holds.

## Claim thinning-preserves-dependence

> In several pairs, position-dependent transmission remains after spikes are thinned so that both cells fire at a flat rate over the region examined.

Methods, "Controlling for power": in the region, spikes are removed one at a time, chosen at random within [-s, +s] (s = 0.05) around the PSTH maximum, until the PSTH minimum is within 10% of the maximum, for both cells; then the position test (as above, all trials). Pairs: the position-dependent connections of fig4-position-dependence, trials (b), (i). Regions (a) 0.2 to 0.5 (Supplementary Figure 13A); (b) the smallest span holding the pair's significant region, extended by an equal length of maze next to it (to the right, or to the left where the maze ends). Ten repetitions with fixed seeds; a pair keeps its dependence in a repetition when a globally significant region meets the region and part of the region has no pointwise break.

In [ ]:
class Thinned:
    """A session view with two cells' spike trains replaced."""
    def __init__(self, x, a, b, ta, tb):
        self.__dict__.update(x.__dict__)
        self.times = list(x.times)
        self.times[a], self.times[b] = ta, tb
    trial_of = fj.Session.trial_of
    position = fj.Session.position
def thin(x, u, lo, hi, rng):
    t = x.times[u]
    k = x.trial_of(t)
    pos = x.position(t)
    inreg = (k >= 0) & (pos >= lo) & (pos <= hi)
    g = (fj.GRID >= lo) & (fj.GRID <= hi)
    K = fj.kernel_matrix(pos[inreg])[:, g]
    keep = np.ones(inreg.sum(), bool)
    P = K.sum(0)
    xs = fj.GRID[g]
    while P.max() > 0 and P.min() < 0.9 * P.max():
        xm = xs[np.argmax(P)]
        cand = np.nonzero(keep & (np.abs(pos[inreg] - xm) <= fj.SIGMA))[0]
        if len(cand) == 0:
            break
        j = rng.choice(cand)
        keep[j] = False
        P = P - K[j]
    full = np.ones(len(t), bool)
    full[np.nonzero(inreg)[0][~keep]] = False
    return t[full], (~keep).sum(), inreg.sum()
DEP = [(s, a, b) for s, a, b in EXC if posdep(POS[(s, a, b, "all")], "i")]
THIN = {}
t1 = time.time()
for rd in "ab":
    for q, (s, a, b) in enumerate(DEP):
        x = S[s]
        segs = POS[(s, a, b, "all")]["segments"]
        if rd == "a":
            lo, hi = 0.2, 0.5
        else:
            p0, p1 = min(g_[0] for g_ in segs), max(g_[1] for g_ in segs)
            L = p1 - p0 + 0.01
            lo, hi = (p0, min(1, p1 + L)) if p1 + L <= 1 else (max(0, p0 - L), p1)
        kept = []
        for rep in range(10):
            rng = np.random.default_rng(70000 + 1000 * q + rep + (0 if rd == "a" else 500))
            ta, ra, na = thin(x, a, lo, hi, rng)
            tb, rb, nb_ = thin(x, b, lo, hi, rng)
            r = fj.coincidence_position_test(Thinned(x, a, b, ta, tb), a, b, np.ones(x.n_trials, bool), M=5000, alpha=0.01, seed=80000 + q * 10 + rep)
            if r is None:
                kept.append(False); continue
            g = (fj.GRID >= lo) & (fj.GRID <= hi)
            kept.append(bool(r["glob"][g].any() and (r["sign"][g] == 0).any()))
        THIN[(rd, s, a, b)] = (np.mean(kept), lo, hi)
print(f"{len(DEP)} position-dependent pairs; thinning took {time.time() - t1:.0f} s")
for rd in "ab":
    v = [(k[1], S[k[1]].cells[k[2]], S[k[1]].cells[k[3]], round(f, 1), round(lo, 2), round(hi, 2)) for k, (f, lo, hi) in THIN.items() if k[0] == rd]
    n_keep = sum(f > 0.5 for *_, f, lo, hi in v)
    print(f"region reading ({rd}): pairs keeping dependence in most repetitions {n_keep} of {len(v)} (at least 2: {n_keep >= 2})")
    print("   ", [(s, pre, post, f) for s, pre, post, f, lo, hi in v if f > 0])

**Result: reproduced (resting on an assumption).** With region 0.2 to 0.5, 7 of the 34 position-dependent connections keep a globally significant region and an unbroken part of the region in most of ten thinnings, among them pair 156-136 (10 of 10). With the region taken from each pair's own significant span, none does.

## Claim fig6-facilitation-depression

> Comparing first spikes of trains with later spikes, about equal shares of pairs show depressing (12.7%) and facilitating (10.7%) spike transmission; 32.2% of the significant pairs are on different shanks.

Presynaptic spikes labelled first (more than 200 ms after the previous spike) or later (less than 40 ms after the previous spike); readings (x) a later spike must belong to a train begun by a first spike, (y) any spike within 40 ms of the previous. For each label the standardized peak height (h minus the jitter mean over the jitter s.d.; Supplementary Figure 9B), with h (i) the largest 1 ms bin in [1, 4] ms or (ii) the sum over [1, 4] ms; the jitter of the shared method (every spike of both cells on [-5, +5] ms, 1,000 surrogates; the presynaptic jitter of Supplementary Figure 9A, 200 surrogates, as a variant); the difference first minus later, labels shuffled 1,000 times, two-sided P < 0.10. Depression is first above later relative to the permutation distribution: a significant pair is depressing when its difference lies in the upper tail (above the permutation median) and facilitating in the lower tail, as the Figure 6c legend assigns the two sides ("two-sided test; one side corresponds 'facilitation', the other to 'depression'"). The permutation distribution is not centred on 0, as the standardized height grows with the number of spikes and first spikes usually outnumber later ones, so the sign of the raw difference (the round-1 labelling, shown as a variant) does not say which side a pair is on. Pairs (a) the excitatory connections, (b) all connections (B-a, W-a). Pairs with fewer than 5 spikes of either label are not tested.

In [ ]:
F6, F6P = {}, {}
t1 = time.time()
for q, (s, a, b, k) in enumerate(CONN[MAIN]):
    ta, tb = S[s].times[a], S[s].times[b]
    for rd in "xy":
        first, later = fj.spike_labels(ta, rd)
        # the shared method's jitter (both cells, 1,000 surrogates), and the presynaptic jitter of Supplementary Fig. 9A (200) as a variant
        F6[(s, a, b, k, rd)] = fj.standardized_height_test(ta, tb, first, later, M_jit=1000, R=1000, seed=90000 + q, jitter="both")
        F6P[(s, a, b, k, rd)] = fj.standardized_height_test(ta, tb, first, later, M_jit=200, R=1000, seed=90000 + q, jitter="pre")
print(f"first/later tests on {len(CONN[MAIN])} connections in {time.time() - t1:.0f} s")
def f6_table(F, label, by_sign=False):
    res = {}
    for pairs_rd in "ab":
        for rd in "xy":
            for pk, pkn in (("max", "i"), ("sum", "ii")):
                rows = [(key, v[pk]) for key, v in F.items() if key[4] == rd and (pairs_rd == "b" or key[3] == "E") and v is not None and v.get(pk) is not None]
                ntested = len(rows)
                nall = sum(1 for key in F if key[4] == rd and (pairs_rd == "b" or key[3] == "E"))
                if by_sign:     # round 1 labelling, by the sign of the raw difference
                    dep = [key for key, r in rows if r["p"] < 0.10 and r["diff"] > 0]
                    fac = [key for key, r in rows if r["p"] < 0.10 and r["diff"] <= 0]
                else:           # by the tail of the permutation distribution (Figure 6c legend)
                    dep = [key for key, r in rows if r["p"] < 0.10 and r["side"] == "dep"]
                    fac = [key for key, r in rows if r["p"] < 0.10 and r["side"] == "fac"]
                sig = dep + fac
                diff_sh = sum(S[key[0]].shank[key[1]] != S[key[0]].shank[key[2]] for key in sig)
                fd, ff, fs = pct(len(dep), ntested), pct(len(fac), ntested), pct(diff_sh, len(sig))
                pd_ = st.binomtest(len(dep), ntested, 0.05, alternative="greater").pvalue
                pf_ = st.binomtest(len(fac), ntested, 0.05, alternative="greater").pvalue
                res[(pairs_rd, rd, pkn)] = (fd, ff, fs, pd_, pf_)
                print(f"{label} pairs ({pairs_rd}), later ({rd}), peak ({pkn}): tested {ntested} of {nall}; depressing {len(dep)} = {fd:.1f}% (12.7 +/- 1.27: {abs(fd-12.7)<=1.27}), "
                      f"facilitating {len(fac)} = {ff:.1f}% (10.7 +/- 1.07: {abs(ff-10.7)<=1.07}), on different shanks {fs:.1f}% of significant (32.2 +/- 3.22: {abs(fs-32.2)<=3.22}); "
                      f"above 5% (one-sided binomial p): {pd_:.2g}, {pf_:.2g}; all three in tolerance: {abs(fd-12.7)<=1.27 and abs(ff-10.7)<=1.07 and abs(fs-32.2)<=3.22}")
    print(f"{label}: combinations with both fractions above 5% at p < 0.05:", [c_ for c_, r in res.items() if r[3] < 0.05 and r[4] < 0.05])
    return res
F6RES = f6_table(F6, "[both cells, 1,000 jitters]")
F6PRES = f6_table(F6P, "[variant: presynaptic jitter, 200]")
f6_table(F6P, "[round 1 labelling by sign, presynaptic jitter]", by_sign=True)
F6SRES = f6_table(F6, "[labelling by sign, shared jitter (both cells, 1,000): the criterion as written]", by_sign=True)
# the permutation null of the first-minus-later difference is not centred on 0
for nm, F in (("both", F6), ("pre", F6P)):
    v = [r["max"] for r in F.values() if r is not None and r.get("max")]
    print(f"[{nm}] peak (i), {len(v)} tests: raw difference > 0 in {np.mean([r['diff'] > 0 for r in v]):.2f}; permutation median > 0 in "
          f"{np.mean([r['null_median'] > 0 for r in v]):.2f} (median of null medians {np.median([r['null_median'] for r in v]):.2f}); "
          f"observed above the null median in {np.mean([r['side'] == 'dep' for r in v]):.2f} "
          f"(two-sided binomial p against 0.5: {st.binomtest(sum(r['side'] == 'dep' for r in v), len(v), 0.5).pvalue:.2g}, n = {len(v)})")
print("significant pairs, both cells, (a, x, i): (session, pre, post) diff / null median / p / side")
for key, v in F6.items():
    if key[3] == "E" and key[4] == "x" and v and v.get("max") and v["max"]["p"] < 0.10:
        r = v["max"]; print(f"   {key[:3]} {r['diff']:.2f} / {r['null_median']:.2f} / {r['p']:.3f} / {r['side']}", end=";")
print()

**Result: not reproduced.** With the shared method's jitter and each significant pair assigned to the tail of its permutation distribution, no combination has all three values within tolerance: depressing pairs are 4.0% to 5.9% and facilitating pairs 6.8% to 12.5%. Facilitation is within 10.7 +/- 1.07 in three combinations ((a) y i 10.4%, (b) y i 9.9%, (b) y ii 11.1%) and depression in none. Facilitation is above the 5% null level in every combination (binomial p from 0.036 to 1.5e-10), depression in none (p = 0.18 or more), so the criterion's partial outcome (both fractions above 5%) is not met. The permutation median of the first-minus-later difference is above 0 in 76% of the tests (median 0.65), which is why the raw sign is not used; the presynaptic jitter variant and the round-1 labelling by sign also give no combination with both fractions above 5% at p < 0.05. The criterion as written (labelling by the sign of the raw difference) with the shared jitter gives the same outcome: depressing 5.3% to 7.7% and facilitating 5.0% to 11.3%, depression above 5% at p < 0.05 in three combinations ((a) x ii, (b) x i, (b) x ii) and facilitation in the four with later spikes (y), with neither in (a) x i, so again no combination has both. The outcome is therefore the same under both labellings. The tail labelling itself leans to facilitation: the observed difference lies above the permutation median in 43% of the tests (binomial p = 3.8e-06 against 50%), so part of the facilitation excess under that labelling may come from the test.

## Claim fig6b-convergent-examples

> Two putative pyramidal cells converging on one layer 2/3 interneuron behave oppositely; cell 197 depresses with spike repetition while the first spikes of cell 186 are ineffective and its later spikes drive the interneuron.

The measure of fig6-facilitation-depression: 197 is depressing when its difference lies above the permutation median, 186 facilitating when below; the literal sign of the raw difference is printed too.

In [ ]:
x = S["EE188"]; c = list(x.cells)
for nm, F in (("both cells, 1,000", F6), ("presynaptic, 200", F6P)):
    for pre in (197, 186):
        a, b = c.index(pre), c.index(201)
        for kk in [kk for kk in F if kk[0] == "EE188" and kk[1] == a and kk[2] == b]:
            v = F[kk]
            print(f"[{nm}] {pre} -> 201 {kk[3]} later ({kk[4]}):",
                  {pk: {m: (round(float(v[pk][m]), 3) if m != "side" else v[pk][m]) for m in v[pk]} if v and v.get(pk) else None for pk in ("max", "sum")},
                  "n first/later", (v["n_first"], v["n_later"]) if v else None)
def dir_ok(pre, side, F):
    a, b = c.index(pre), c.index(201)
    out = []
    for kk, v in F.items():
        if kk[0] == "EE188" and kk[1] == a and kk[2] == b and v:
            for pk in ("max", "sum"):
                if v.get(pk):
                    out.append((kk[4], pk, v[pk]["side"] == side, v[pk]["p"] < 0.10))
    return out
for nm, F in (("both cells, 1,000", F6), ("presynaptic, 200", F6P)):
    d197, d186 = dir_ok(197, "dep", F), dir_ok(186, "fac", F)
    print(f"[{nm}] 197 depressing side (side, P < 0.10) per reading:", d197)
    print(f"[{nm}] 186 facilitating side (side, P < 0.10) per reading:", d186)
    print(f"   197 depressing and significant under a reading: {any(a_ and b_ for *_, a_, b_ in d197)}; 186 facilitating and significant: {any(a_ and b_ for *_, a_, b_ in d186)}; "
          f"both on their sides under some reading: {any(a_ for *_, a_, b_ in d197) and any(a_ for *_, a_, b_ in d186)}")

**Result: partially reproduced.** By the tail of the permutation distribution, 186 -> 201 is on the facilitating side under all four readings and significant at P < 0.10 under three ((x, max) p = 0.016, (x, sum) p = 0.050, (y, sum) p = 0.060); 197 -> 201 is on the depressing side only under (x, max), not significantly (p = 0.875). By the literal sign of the raw difference, 197 has the first spikes above the later under every reading (13.8 against 4.3, (x, max)) but its difference is close to the permutation median (9.54 against 9.25). Both are on their sides under some reading and only 186 is significant, which is the criterion's partial outcome; with the presynaptic jitter 197 is on the facilitating side under every reading.

## Claim fig7-supralinear

> Coincident spikes of two or more presynaptic cells within 5 ms discharge their common target interneuron supralinearly, more than the sum of their separate effects, while coincidences in windows over 10 ms add only linearly; group data come from 14 satellites.

A satellite: a putative interneuron (B-a, W-a) with at least two (S-a) or three (S-b) excitatory presynaptic putative pyramidal cells. Events: the pooled spikes of the satellite's presynaptic cells are grouped into events, an event starting at a spike and holding every pooled spike within the window after it (2, 5, 10, 20 ms); k is the number of spikes in the event, or of distinct presynaptic cells (both run). Transmission probability: the fraction of events followed by a target spike 1 to 4 ms after the last presynaptic spike of the event (after the first, as a variant). Linear predictions (a) k p1, (b) 1 - (1 - p1)^k.

In [ ]:
def satellites(min_pre):
    out = []
    for s, x in S.items():
        for u in range(len(x.cells)):
            if TY[(s, u)] != "int":
                continue
            pres = sorted({a for ss, a, b, k in CONN[MAIN] if ss == s and b == u and k == "E" and TY[(s, a)] == "pyr"})
            if len(pres) >= min_pre:
                out.append((s, u, pres))
    return out
def transmission(s, u, pres, win, count="spikes", anchor="last"):
    x = S[s]
    t = np.concatenate([x.times[a] for a in pres]); who = np.concatenate([[a] * len(x.times[a]) for a in pres])
    o = np.argsort(t); t, who = t[o], who[o]
    tgt = x.times[u]
    i, n = 0, len(t)
    ks, hit = [], []
    while i < n:
        j = i
        while j + 1 < n and t[j + 1] - t[i] <= win:
            j += 1
        k = (j - i + 1) if count == "spikes" else len(set(who[i:j + 1]))
        ref = t[j] if anchor == "last" else t[i]
        lo_, hi_ = np.searchsorted(tgt, [ref + 0.001, ref + 0.004])
        ks.append(k); hit.append(hi_ > lo_)
        i = j + 1
    ks, hit = np.array(ks), np.array(hit)
    return {k: (hit[ks == k].mean(), int((ks == k).sum())) for k in range(1, 5) if (ks == k).sum() >= 5}
for min_pre, nm in ((2, "S-a"), (3, "S-b")):
    sats = satellites(min_pre)
    print(f"{nm}: satellites {len(sats)} (14 +/- 1.4: {abs(len(sats)-14)<=1.4})", [(s, S[s].cells[u], len(p)) for s, u, p in sats])
    for count in ("spikes", "cells"):
        for anchor in ("last", "first"):
            summ = {}
            for win in (0.002, 0.005, 0.010, 0.020):
                rows = [transmission(s, u, p, win, count, anchor) for s, u, p in sats]
                for lin in "ab":
                    for k in (2, 3):
                        vals = []
                        for r in rows:
                            if 1 in r and k in r:
                                p1 = r[1][0]
                                pred = k * p1 if lin == "a" else 1 - (1 - p1) ** k
                                vals.append(r[k][0] - pred)
                        summ[(win, lin, k)] = (np.mean(vals) if vals else np.nan, np.std(vals) if vals else np.nan,
                                               int(np.sum(np.array(vals) > 0)), len(vals))
            for lin in "ab":
                s5 = [summ[(0.005, lin, k)] for k in (2, 3)]
                s20 = [summ[(0.020, lin, k)] for k in (2, 3)]
                ok5 = all(v[3] > 0 and v[0] > 0 and v[2] > v[3] / 2 for v in s5)
                ok20 = all(not (v[0] > v[1]) for v in s20 if v[3] > 0)
                print(f"   k={count}, lag after {anchor} spike, linear ({lin}): excess over linear, mean (s.d.; satellites above / with data) "
                      f"5 ms k=2 {s5[0][0]:.3f} ({s5[0][1]:.3f}; {s5[0][2]}/{s5[0][3]}), k=3 {s5[1][0]:.3f} ({s5[1][1]:.3f}; {s5[1][2]}/{s5[1][3]}); "
                      f"20 ms k=2 {s20[0][0]:.3f} ({s20[0][1]:.3f}), k=3 {s20[1][0]:.3f} ({s20[1][1]:.3f}) | 5 ms supralinear {ok5}, 20 ms not above by more than s.d. {ok20}")

**Result: not reproduced.** With satellites of three or more presynaptic cells there are 14 (14 +/- 1.4), but with 5 ms windows the transmission probability after 2 and 3 coincident spikes is below both linear predictions in almost every satellite (0 of 14 above for 2 spikes), under every reading of k and of the lag. The 20 ms part (not above the prediction) holds.

## Summary

| Claim | Outcome |
|---|---|
| channel-region-link | not reproduced (GG has no nose-poke sensor on channels 97 and 98) |
| rats-and-units | partially reproduced (assumed: unit list) |
| task-performance | blocked, data_missing |
| fig2-population-uniform | not reproduced |
| fig2c-active-fraction | partially reproduced |
| fig2-field-size | reproduced |
| population-vector-width | partially reproduced (assumed) |
| fig2d-layer-rates | partially reproduced |
| fig2e-layer-discrimination | not reproduced |
| traj-diverge | partially reproduced (assumed) |
| side-arm-selectivity-by-layer | partially reproduced |
| central-arm-selectivity | partially reproduced (assumed: unit list) |
| odor-nosepoke | not reproduced |
| odor-unrelated-to-stem | not reproduced |
| retrospective-coding-rare | reproduced |
| speed-not-correlated | reproduced |
| fig3-connection-rates | not reproduced |
| fig3-cell-classes | not reproduced |
| pyr-pyr-connections | partially reproduced |
| nearby-pyr-int-probability | not reproduced |
| fig3b-session-connections | not reproduced |
| fig3d-hub | not reproduced |
| fig3f-distance | partially reproduced |
| fig3-trajectory-by-class | partially reproduced |
| no-spatial-clustering | reproduced (layer 2/3 session only) |
| fig4-position-dependence | partially reproduced (assumed) |
| fig5a-session-selective-connections | not reproduced (assumed) |
| fig4-pair-156-136 | not reproduced (assumed) |
| fig5b-example-pairs | not reproduced (assumed) |
| thinning-preserves-dependence | reproduced (assumed) |
| fig6-facilitation-depression | not reproduced |
| fig6b-convergent-examples | partially reproduced |
| fig7-supralinear | not reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

**How the analysis differs from the paper's.** The datasets hold three rats, not four, and give no rat's layer, so every layer comparison is run under each assignment of the layer 5 rat. Only correct trials are listed, and position 0 and 1 are the lab's (end of the last poke, start of reward consumption). The layer 5 session of Figure 2a and Supplementary Figure 11 (142 cells) is not in the datasets; FF.130 is shown as a stand-in. Units are the lab's cell lists (627; for EE.188 the 117 cells of its example file), an assumption: they are consistent with the 1 Hz "Cells" list of the CRCNS description, and every sorted cluster of the NWB files (1,373 units) is shown as a variant for the fractions, the active fraction and the rates; the rats-and-units and central-arm outcomes would change with it (rats-and-units to not reproduced, as the total of 1,373 is outside tolerance). The number of permutations or surrogates is not given for the left-right test (1,000 used), the trajectory test (10,000) and the position test of coincidences (5,000); at level 0.01 the discrete global band needs more than 1,000. The first/later spike test uses the shared method's jitter (both cells, 1,000 surrogates; presynaptic jitter as a variant) and assigns a significant pair to depression or facilitation by the tail of its permutation distribution, as the Figure 6c legend does. The population vector width needs a normalisation of each neuron's rate that the paper does not state; the peak normalisation of Figure 2a is assumed. The jitter test of connections finds about twice the paper's rate of excitatory connections and few cells with a trough alone, so the cell classes, and the claims built on them, differ from the paper's.

**What was found about the datasets.** DANDI:000067 has no wideband signal for EE.088, no trials or linearised position for EE.188 and EE.210, and no cell types or lab cell numbers for 11 of 13 sessions, though CRCNS pfc-2 holds the lab's cell lists for all of them; its units can be linked to the lab's cells only through shank and cluster, and then their spike times are identical (615 cells; the 12 cells of EE.188's example file that the behavior file does not list have no lab spike times to compare). In GG, channels 97 and 98 carry no nose-poke signal, contrary to the CRCNS description; the NWB files label channels 97 to 101 'neural'. FF has 104 channels in the NWB files, not the 101 of the description. The two CRCNS cell lists of EE.188 disagree (the behavior file's 105 cells, 87 pyramidal and 18 interneurons, against the example file's 117 cells, 33, 15 and 69 unknown; types agree for 42 of 105), and the example file labels cells 32 and 72 interneurons and 136 unknown, where the paper names 32 and 72 presynaptic pyramidal cells and 136 an interneuron. The example file's connection list reproduces the paper's hub of 39 cells. Error trials are not listed, so task performance cannot be computed. Ten of 13 sessions have one tracked LED, so head direction exists in three. The CellsInfo lists of the behavior files, the cell lists used, are not described in the CRCNS description, which documents a "Cells" variable that the files do not hold. The linearised position is nonzero (about 0.02 to 0.05) during each trial's nose poke, which the description does not say; these frames are the 18% to 67% of frames with d > 0 outside the listed trials.

In [ ]:
print(f"notebook ran in {(time.time() - T0) / 60:.1f} min; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")